<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/08_SPP_GAN_Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# NOTEBOOK 08 — SPP-GAN ARCHITECTURE
# Research: A Unified Privacy-Preserving Framework for High-Fidelity Synthetic Data Generation
# Framework: SPP-GAN
# ==================================================================================================
#
# PURPOSE
# -------
# Define, validate, document, and persist the complete SPP-GAN neural architecture.
#
# This notebook:
#   1. Loads the canonical project configuration and feature schemas.
#   2. Loads Notebook 03 statistical profiles when available.
#   3. Defines the SPP-GAN input representation.
#   4. Defines latent space.
#   5. Defines Generator.
#   6. Defines Discriminator/Critic.
#   7. Defines numerical and categorical output mechanisms.
#   8. Defines interfaces for statistical guidance and privacy.
#   9. Initializes one architecture per registered dataset.
#  10. Performs parameter, tensor, forward, backward, and gradient tests.
#  11. Saves architecture/configuration artifacts for downstream notebooks.
#
# IMPORTANT
# ---------
# This is an ARCHITECTURE notebook.
#
# No model training is performed here.
# No privacy budget is consumed here.
# No synthetic data is generated here.
#
# Downstream:
#   Notebook 09 -> Statistical Guidance
#   Notebook 10 -> Differential Privacy
#   Notebook 11 -> Privacy Accounting
#   Notebook 12 -> SPP-GAN Training
#   Notebook 13 -> Synthetic Data Generation
#
# RAM DESIGN
# ----------
#   • Process one dataset at a time.
#   • Never load all datasets simultaneously.
#   • Never duplicate full training datasets.
#   • Architecture tests use very small tensors.
#   • Persist compact metadata rather than large tensors.
#
# ==================================================================================================

In [2]:
# ==================================================================================================
# 1. HEADER & SCOPE
# ==================================================================================================

from pathlib import Path
import os
import sys
import json
import math
import time
import hashlib
import platform
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 100)
print("NOTEBOOK 08 — SPP-GAN ARCHITECTURE")
print("=" * 100)

print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
else:
    print("GPU          : CPU")

print("=" * 100)

# -----------------------------------------------------------------------------------------------
# Scope
# -----------------------------------------------------------------------------------------------

NOTEBOOK_ID = "08"
NOTEBOOK_NAME = "SPP-GAN Architecture"
FRAMEWORK_NAME = "SPP-GAN"

RESEARCH_TITLE = (
    "A Unified Privacy-Preserving Framework for High-Fidelity "
    "Synthetic Data Generation Using Statistical and Machine Learning Models"
)

MASTER_SEED = 2025

# Architecture notebook does not train.
TRAINING_ENABLED = False
PRIVACY_ENABLED = False
SYNTHETIC_GENERATION_ENABLED = False

print(f"Framework                : {FRAMEWORK_NAME}")
print(f"Research title           : {RESEARCH_TITLE}")
print(f"Master seed              : {MASTER_SEED}")
print(f"Training enabled         : {TRAINING_ENABLED}")
print(f"Privacy mechanism active : {PRIVACY_ENABLED}")
print(f"Synthetic generation     : {SYNTHETIC_GENERATION_ENABLED}")

NOTEBOOK 08 — SPP-GAN ARCHITECTURE
Python       : 3.13.15
PyTorch      : 2.11.0+cpu
CUDA         : False
GPU          : CPU
Framework                : SPP-GAN
Research title           : A Unified Privacy-Preserving Framework for High-Fidelity Synthetic Data Generation Using Statistical and Machine Learning Models
Master seed              : 2025
Training enabled         : False
Privacy mechanism active : False
Synthetic generation     : False


In [3]:
# ==================================================================================================
# 2. LOAD CONFIGURATION
# ==================================================================================================

import yaml

# -----------------------------------------------------------------------------------------------
# Mount Google Drive FIRST
# -----------------------------------------------------------------------------------------------

try:
    from google.colab import drive

    drive_root = Path("/content/drive")

    if not drive_root.exists() or not (drive_root / "MyDrive").exists():
        drive.mount("/content/drive")
        print("✓ Google Drive mounted.")
    else:
        print("✓ Google Drive already mounted.")

except ImportError:
    print("ℹ Google Colab not detected. Continuing with existing filesystem.")

# -----------------------------------------------------------------------------------------------
# Canonical project root
# -----------------------------------------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Canonical project root not found after Drive initialization:\n"
        f"{PROJECT_ROOT}"
    )

print(f"✓ Project root : {PROJECT_ROOT}")

# -----------------------------------------------------------------------------------------------
# Notebook 08 directories
# -----------------------------------------------------------------------------------------------

NB08_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_08"

DIRS = {
    "root": NB08_ROOT,
    "architecture": NB08_ROOT / "architecture",
    "config": NB08_ROOT / "config",
    "validation": NB08_ROOT / "validation",
    "metadata": NB08_ROOT / "metadata",
    "models": NB08_ROOT / "models",
    "logs": NB08_ROOT / "logs",
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print(f"✓ Notebook 08 root : {NB08_ROOT}")

# -----------------------------------------------------------------------------------------------
# Notebook 00 root
# -----------------------------------------------------------------------------------------------

NB00_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_00"

if not NB00_ROOT.exists():
    raise FileNotFoundError(
        f"Notebook 00 output directory not found:\n"
        f"{NB00_ROOT}"
    )

print(f"✓ Notebook 00 : {NB00_ROOT}")

# -----------------------------------------------------------------------------------------------
# Authoritative Notebook 00 experiment configuration
# -----------------------------------------------------------------------------------------------

CONFIG_PATH = NB00_ROOT / "config" / "experiment_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        "Authoritative Notebook 00 experiment configuration not found:\n"
        f"{CONFIG_PATH}"
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

if not isinstance(CONFIG, dict):
    raise TypeError(
        "Notebook 00 experiment configuration must be a JSON object."
    )

print(f"✓ Configuration loaded : {CONFIG_PATH}")

# -----------------------------------------------------------------------------------------------
# Configuration fingerprint
# -----------------------------------------------------------------------------------------------

CONFIG_FINGERPRINT_PATH = (
    NB00_ROOT
    / "manifest"
    / "configuration_fingerprint.json"
)

if CONFIG_FINGERPRINT_PATH.exists():

    with open(CONFIG_FINGERPRINT_PATH, "r", encoding="utf-8") as f:
        CONFIG_FINGERPRINT = json.load(f)

    print(
        f"✓ Configuration fingerprint loaded : "
        f"{CONFIG_FINGERPRINT_PATH}"
    )

else:

    CONFIG_FINGERPRINT = None

    print(
        "⚠ Configuration fingerprint not found. "
        "Continuing without fingerprint metadata."
    )

# -----------------------------------------------------------------------------------------------
# Helper to retrieve nested values
# -----------------------------------------------------------------------------------------------

def cfg_get(obj, *keys, default=None):
    current = obj

    for key in keys:
        if not isinstance(current, dict) or key not in current:
            return default

        current = current[key]

    return current

# -----------------------------------------------------------------------------------------------
# Canonical Notebook 00 dataset registry
# -----------------------------------------------------------------------------------------------

DATASET_REGISTRY_PATH = (
    NB00_ROOT
    / "config"
    / "dataset_registry.json"
)

if not DATASET_REGISTRY_PATH.exists():
    raise FileNotFoundError(
        "Canonical Notebook 00 dataset registry not found:\n"
        f"{DATASET_REGISTRY_PATH}"
    )

with open(DATASET_REGISTRY_PATH, "r", encoding="utf-8") as f:
    DATASET_REGISTRY = json.load(f)

if not isinstance(DATASET_REGISTRY, dict):
    raise TypeError(
        "Notebook 00 dataset registry must be a dictionary "
        "keyed by dataset identifier."
    )

# -----------------------------------------------------------------------------------------------
# Validate required research datasets
# -----------------------------------------------------------------------------------------------

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

missing_datasets = [
    dataset
    for dataset in EXPECTED_DATASETS
    if dataset not in DATASET_REGISTRY
]

if missing_datasets:
    raise ValueError(
        "Required datasets are missing from the Notebook 00 dataset registry:\n"
        f"{missing_datasets}"
    )

# Validate dataset configuration records
for dataset_name in EXPECTED_DATASETS:

    record = DATASET_REGISTRY[dataset_name]

    if not isinstance(record, dict):
        raise TypeError(
            f"Dataset registry entry for '{dataset_name}' "
            f"must be a dictionary."
        )

    required_fields = [
        "dataset_id",
        "enabled",
        "target_column",
        "task_type",
    ]

    missing_fields = [
        field
        for field in required_fields
        if field not in record
    ]

    if missing_fields:
        raise ValueError(
            f"Dataset '{dataset_name}' is missing required registry fields:\n"
            f"{missing_fields}"
        )

print(f"✓ Dataset registry loaded : {DATASET_REGISTRY_PATH}")
print(f"✓ Registered datasets     : {len(DATASET_REGISTRY)}")
print(f"  {EXPECTED_DATASETS}")

# -----------------------------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------------------------

torch.manual_seed(MASTER_SEED)
np.random.seed(MASTER_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MASTER_SEED)

print(f"✓ Master seed initialized : {MASTER_SEED}")

Mounted at /content/drive
✓ Google Drive mounted.
✓ Project root : /content/drive/MyDrive/SPP_GAN_Research
✓ Notebook 08 root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08
✓ Notebook 00 : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00
✓ Configuration loaded : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/config/experiment_config.json
✓ Configuration fingerprint loaded : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/manifest/configuration_fingerprint.json
✓ Dataset registry loaded : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/config/dataset_registry.json
✓ Registered datasets     : 3
  ['adult_income', 'bank_marketing', 'diabetes_130us']
✓ Master seed initialized : 2025


In [4]:
# ==================================================================================================
# 3. LOAD FEATURE SCHEMA
# ==================================================================================================

print("=" * 100)
print("3. LOAD FEATURE SCHEMA")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# Canonical Notebook 02 location
# -----------------------------------------------------------------------------------------------

NB02_ROOT = PROJECT_ROOT / "data" / "processed" / "notebook_02"

if not NB02_ROOT.exists():
    raise FileNotFoundError(
        f"Notebook 02 processed directory not found:\n{NB02_ROOT}"
    )

print(f"✓ Notebook 02 root : {NB02_ROOT}")

# -----------------------------------------------------------------------------------------------
# Canonical native manifest
# -----------------------------------------------------------------------------------------------

NATIVE_MANIFEST = (
    NB02_ROOT
    / "native"
    / "native_dataset_manifest.csv"
)

if not NATIVE_MANIFEST.exists():
    raise FileNotFoundError(
        "Canonical Notebook 02 native manifest not found:\n"
        f"{NATIVE_MANIFEST}"
    )

print(f"✓ Native manifest : {NATIVE_MANIFEST}")

native_manifest_df = pd.read_csv(NATIVE_MANIFEST)

print(f"✓ Manifest rows   : {len(native_manifest_df):,}")

# -----------------------------------------------------------------------------------------------
# Validate required manifest columns
# -----------------------------------------------------------------------------------------------

REQUIRED_MANIFEST_COLUMNS = [
    "dataset_id",
    "split",
    "absolute_path",
    "rows",
    "columns",
    "generative_columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "provenance_present",
    "target_present",
    "identifiers_excluded",
    "file_exists",
    "reload_validation",
    "status",
]

missing_manifest_columns = [
    column
    for column in REQUIRED_MANIFEST_COLUMNS
    if column not in native_manifest_df.columns
]

if missing_manifest_columns:
    raise ValueError(
        "Notebook 02 native manifest is missing required columns:\n"
        f"{missing_manifest_columns}"
    )

print("✓ Native manifest schema validated.")

# -----------------------------------------------------------------------------------------------
# Validate expected datasets
# -----------------------------------------------------------------------------------------------

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

manifest_datasets = set(
    native_manifest_df["dataset_id"]
    .astype(str)
    .str.strip()
)

missing_datasets = [
    dataset
    for dataset in EXPECTED_DATASETS
    if dataset not in manifest_datasets
]

if missing_datasets:
    raise ValueError(
        "Required datasets are missing from Notebook 02 native manifest:\n"
        f"{missing_datasets}"
    )

# -----------------------------------------------------------------------------------------------
# Resolve authoritative training manifest record
# -----------------------------------------------------------------------------------------------

def resolve_training_manifest_record(dataset_name):

    matches = native_manifest_df[
        (native_manifest_df["dataset_id"].astype(str).str.strip() == dataset_name)
        & (
            native_manifest_df["split"]
            .astype(str)
            .str.strip()
            .str.lower()
            == "train"
        )
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one training manifest record for "
            f"'{dataset_name}', found {len(matches)}."
        )

    record = matches.iloc[0].to_dict()

    if str(record["status"]).upper() != "PASS":
        raise ValueError(
            f"Notebook 02 training manifest record for "
            f"'{dataset_name}' is not PASS."
        )

    if not bool(record["file_exists"]):
        raise FileNotFoundError(
            f"Notebook 02 training file does not exist for "
            f"'{dataset_name}'."
        )

    if not bool(record["reload_validation"]):
        raise ValueError(
            f"Notebook 02 reload validation failed for "
            f"'{dataset_name}'."
        )

    return record

# -----------------------------------------------------------------------------------------------
# Parse persisted identifier columns
# -----------------------------------------------------------------------------------------------

def parse_identifier_columns(value):

    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    if isinstance(value, list):
        return [str(v).strip() for v in value]

    value = str(value).strip()

    if value in ("", "[]", "None", "nan"):
        return []

    try:
        parsed = json.loads(value)

        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed]

    except (json.JSONDecodeError, TypeError):
        pass

    return [
        item.strip()
        for item in value.split(",")
        if item.strip()
    ]

# -----------------------------------------------------------------------------------------------
# Load authoritative feature schemas
# -----------------------------------------------------------------------------------------------

FEATURE_SCHEMAS = {}

for dataset_name in EXPECTED_DATASETS:

    record = resolve_training_manifest_record(dataset_name)

    train_path = Path(str(record["absolute_path"]))

    if not train_path.exists():
        train_path = (
            NB02_ROOT
            / "native"
            / dataset_name
            / "train.csv"
        )

    if not train_path.exists():
        raise FileNotFoundError(
            f"Canonical training file not found for "
            f"'{dataset_name}':\n{train_path}"
        )

    # -------------------------------------------------------------------------------------------
    # RAM-safe header inspection
    # -------------------------------------------------------------------------------------------

    header_df = pd.read_csv(
        train_path,
        nrows=0,
    )

    native_columns = list(header_df.columns)

    # -------------------------------------------------------------------------------------------
    # Authoritative target
    # -------------------------------------------------------------------------------------------

    target = str(record["target_column"]).strip()

    if target not in native_columns:
        raise ValueError(
            f"Target '{target}' is missing from the native schema "
            f"for '{dataset_name}'."
        )

    if not bool(record["target_present"]):
        raise ValueError(
            f"Notebook 02 reports target '{target}' as absent "
            f"for '{dataset_name}'."
        )

    # -------------------------------------------------------------------------------------------
    # Authoritative provenance
    # -------------------------------------------------------------------------------------------

    provenance_column = str(
        record["provenance_column"]
    ).strip()

    if provenance_column not in native_columns:
        raise ValueError(
            f"Provenance column '{provenance_column}' is missing "
            f"from the native schema for '{dataset_name}'."
        )

    # -------------------------------------------------------------------------------------------
    # Authoritative identifiers
    # -------------------------------------------------------------------------------------------

    identifiers = parse_identifier_columns(
        record["identifier_columns"]
    )

    identifiers_excluded = bool(
        record["identifiers_excluded"]
    )

    # Identifiers recorded by Notebook 02 must NOT be part of
    # the modeling/generative schema.
    identifiers_present_in_native = [
        identifier
        for identifier in identifiers
        if identifier in native_columns
    ]

    if identifiers_present_in_native:
        raise ValueError(
            f"Identifier columns were expected to be excluded but "
            f"remain in the native schema for '{dataset_name}':\n"
            f"{identifiers_present_in_native}"
        )

    if identifiers and not identifiers_excluded:
        raise ValueError(
            f"Notebook 02 reports identifier columns for "
            f"'{dataset_name}', but identifiers_excluded=False."
        )

    # -------------------------------------------------------------------------------------------
    # Reconstruct generative columns from the persisted native schema
    # -------------------------------------------------------------------------------------------

    excluded_columns = {
        provenance_column,
        *identifiers,
    }

    generative_columns = [
        column
        for column in native_columns
        if column not in excluded_columns
    ]

    # -------------------------------------------------------------------------------------------
    # Cross-check persisted generative dimension
    # -------------------------------------------------------------------------------------------

    persisted_generative_dimension = int(
        record["generative_columns"]
    )

    if len(generative_columns) != persisted_generative_dimension:
        raise ValueError(
            f"Generative schema mismatch for '{dataset_name}': "
            f"manifest={persisted_generative_dimension}, "
            f"reconstructed={len(generative_columns)}."
        )

    # -------------------------------------------------------------------------------------------
    # Target must remain generative
    # -------------------------------------------------------------------------------------------

    if target not in generative_columns:
        raise ValueError(
            f"Target '{target}' is not present in generative columns "
            f"for '{dataset_name}'."
        )

    # -------------------------------------------------------------------------------------------
    # Cross-check native dimension
    # -------------------------------------------------------------------------------------------

    persisted_native_dimension = int(
        record["columns"]
    )

    if len(native_columns) != persisted_native_dimension:
        raise ValueError(
            f"Native schema dimension mismatch for '{dataset_name}': "
            f"manifest={persisted_native_dimension}, "
            f"actual={len(native_columns)}."
        )

    # -------------------------------------------------------------------------------------------
    # Store canonical schema
    # -------------------------------------------------------------------------------------------

    FEATURE_SCHEMAS[dataset_name] = {
        "dataset": dataset_name,
        "train_path": str(train_path),
        "native_columns": native_columns,
        "generative_columns": generative_columns,
        "target": target,
        "identifiers": identifiers,
        "provenance_columns": [provenance_column],
        "native_dimension": len(native_columns),
        "generative_dimension": len(generative_columns),
        "manifest_rows": int(record["rows"]),
        "manifest_columns": persisted_native_dimension,
        "manifest_generative_columns": persisted_generative_dimension,
        "manifest_sha256": str(record["sha256"]),
    }

    print(
        f"✓ {dataset_name:<20} "
        f"native={len(native_columns):>3} | "
        f"generative={len(generative_columns):>3} | "
        f"target={target:<12} | "
        f"identifiers={len(identifiers)}"
    )

# -----------------------------------------------------------------------------------------------
# Final registry validation
# -----------------------------------------------------------------------------------------------

if set(FEATURE_SCHEMAS.keys()) != set(EXPECTED_DATASETS):
    raise RuntimeError(
        "FEATURE_SCHEMAS does not contain exactly the expected datasets."
    )

print("\n✓ Feature schema registry created.")
print("✓ Notebook 02 authoritative schema successfully inherited.")

3. LOAD FEATURE SCHEMA
✓ Notebook 02 root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Native manifest : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/native_dataset_manifest.csv
✓ Manifest rows   : 9
✓ Native manifest schema validated.
✓ adult_income         native= 16 | generative= 15 | target=income       | identifiers=0
✓ bank_marketing       native= 18 | generative= 17 | target=y            | identifiers=0
✓ diabetes_130us       native= 49 | generative= 48 | target=readmitted   | identifiers=2

✓ Feature schema registry created.
✓ Notebook 02 authoritative schema successfully inherited.


In [5]:
# ==================================================================================================
# 4. LOAD STATISTICAL PROFILE
# ==================================================================================================

print("=" * 100)
print("4. LOAD STATISTICAL PROFILE")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# Canonical Notebook 03 location
# -----------------------------------------------------------------------------------------------

NB03_ROOT = PROJECT_ROOT / "data" / "processed" / "notebook_03"

if not NB03_ROOT.exists():
    raise FileNotFoundError(
        f"Notebook 03 processed directory not found:\n{NB03_ROOT}"
    )

print(f"✓ Notebook 03 root : {NB03_ROOT}")

# -----------------------------------------------------------------------------------------------
# Canonical statistical guidance and reference directories
# -----------------------------------------------------------------------------------------------

GUIDANCE_DIR = NB03_ROOT / "guidance"
REFERENCE_DIR = NB03_ROOT / "reference"

if not GUIDANCE_DIR.exists():
    raise FileNotFoundError(
        f"Notebook 03 guidance directory not found:\n{GUIDANCE_DIR}"
    )

if not REFERENCE_DIR.exists():
    raise FileNotFoundError(
        f"Notebook 03 reference directory not found:\n{REFERENCE_DIR}"
    )

# -----------------------------------------------------------------------------------------------
# Required SPP-GAN statistical artifact paths
# -----------------------------------------------------------------------------------------------

STATISTICAL_GUIDANCE_PATHS = {
    dataset_name: GUIDANCE_DIR / (
        f"{dataset_name}_spp_gan_statistical_guidance.json"
    )
    for dataset_name in EXPECTED_DATASETS
}

STATISTICAL_REFERENCE_PATHS = {
    dataset_name: REFERENCE_DIR / (
        f"{dataset_name}_spp_gan_statistical_reference.json"
    )
    for dataset_name in EXPECTED_DATASETS
}

# -----------------------------------------------------------------------------------------------
# Validate artifact existence
# -----------------------------------------------------------------------------------------------

for dataset_name in EXPECTED_DATASETS:

    guidance_path = STATISTICAL_GUIDANCE_PATHS[dataset_name]
    reference_path = STATISTICAL_REFERENCE_PATHS[dataset_name]

    if not guidance_path.exists():
        raise FileNotFoundError(
            f"SPP-GAN statistical guidance artifact not found for "
            f"'{dataset_name}':\n{guidance_path}"
        )

    if not reference_path.exists():
        raise FileNotFoundError(
            f"SPP-GAN statistical reference artifact not found for "
            f"'{dataset_name}':\n{reference_path}"
        )

# -----------------------------------------------------------------------------------------------
# Required guidance schema
# -----------------------------------------------------------------------------------------------

REQUIRED_GUIDANCE_KEYS = [
    "guidance_version",
    "guidance_type",
    "source_reference_version",
    "source_reference_type",
    "feature_schema",
    "numeric_feature_guidance",
    "categorical_feature_guidance",
    "strongest_numeric_pearson_dependencies",
    "strongest_numeric_spearman_dependencies",
    "strongest_categorical_dependencies",
    "target_policy",
    "identifier_policy",
    "provenance_policy",
    "evidence_policy",
]

# -----------------------------------------------------------------------------------------------
# Required reference schema
# -----------------------------------------------------------------------------------------------

REQUIRED_REFERENCE_KEYS = [
    "reference_version",
    "reference_type",
    "dataset_id",
    "creation_timestamp_utc",
    "random_seed",
    "fit_policy",
    "schema_policy",
    "dataset_profile",
    "feature_schema",
    "feature_profiles",
    "pearson_correlation",
    "spearman_correlation",
    "categorical_dependency",
]

# -----------------------------------------------------------------------------------------------
# Load authoritative statistical profiles
# -----------------------------------------------------------------------------------------------

STATISTICAL_PROFILE = {}

for dataset_name in EXPECTED_DATASETS:

    guidance_path = STATISTICAL_GUIDANCE_PATHS[dataset_name]
    reference_path = STATISTICAL_REFERENCE_PATHS[dataset_name]

    # -------------------------------------------------------------------------------------------
    # Load guidance
    # -------------------------------------------------------------------------------------------

    with open(
        guidance_path,
        "r",
        encoding="utf-8",
    ) as f:
        guidance = json.load(f)

    if not isinstance(guidance, dict):
        raise TypeError(
            f"Statistical guidance for '{dataset_name}' "
            f"must be a JSON object."
        )

    missing_guidance_keys = [
        key
        for key in REQUIRED_GUIDANCE_KEYS
        if key not in guidance
    ]

    if missing_guidance_keys:
        raise ValueError(
            f"Statistical guidance for '{dataset_name}' is missing "
            f"required fields:\n{missing_guidance_keys}"
        )

    # -------------------------------------------------------------------------------------------
    # Load reference
    # -------------------------------------------------------------------------------------------

    with open(
        reference_path,
        "r",
        encoding="utf-8",
    ) as f:
        reference = json.load(f)

    if not isinstance(reference, dict):
        raise TypeError(
            f"Statistical reference for '{dataset_name}' "
            f"must be a JSON object."
        )

    missing_reference_keys = [
        key
        for key in REQUIRED_REFERENCE_KEYS
        if key not in reference
    ]

    if missing_reference_keys:
        raise ValueError(
            f"Statistical reference for '{dataset_name}' is missing "
            f"required fields:\n{missing_reference_keys}"
        )

    # -------------------------------------------------------------------------------------------
    # Validate dataset identity
    # -------------------------------------------------------------------------------------------

    reference_dataset_id = str(
        reference["dataset_id"]
    ).strip()

    if reference_dataset_id != dataset_name:
        raise ValueError(
            f"Statistical reference dataset mismatch: "
            f"expected='{dataset_name}', "
            f"found='{reference_dataset_id}'."
        )

    # -------------------------------------------------------------------------------------------
    # Validate feature schema against Notebook 08 schema
    # -------------------------------------------------------------------------------------------

    guidance_schema = guidance["feature_schema"]
    reference_schema = reference["feature_schema"]

    if not isinstance(guidance_schema, dict):
        raise TypeError(
            f"Guidance feature_schema for '{dataset_name}' "
            f"must be a dictionary."
        )

    if not isinstance(reference_schema, dict):
        raise TypeError(
            f"Reference feature_schema for '{dataset_name}' "
            f"must be a dictionary."
        )

    # -------------------------------------------------------------------------------------------
    # Validate generative feature count
    # -------------------------------------------------------------------------------------------

    expected_generative_dimension = FEATURE_SCHEMAS[
        dataset_name
    ]["generative_dimension"]

    # Attempt to obtain the persisted feature list where available.
    guidance_feature_list = (
        guidance_schema.get("generative_columns")
        or guidance_schema.get("feature_columns")
        or guidance_schema.get("columns")
    )

    if guidance_feature_list is not None:

        if not isinstance(guidance_feature_list, list):
            raise TypeError(
                f"Guidance feature list for '{dataset_name}' "
                f"must be a list."
            )

        if len(guidance_feature_list) != expected_generative_dimension:
            raise ValueError(
                f"Guidance feature dimension mismatch for "
                f"'{dataset_name}': "
                f"expected={expected_generative_dimension}, "
                f"found={len(guidance_feature_list)}."
            )

    # -------------------------------------------------------------------------------------------
    # Store complete authoritative statistical profile
    # -------------------------------------------------------------------------------------------

    STATISTICAL_PROFILE[dataset_name] = {
        "dataset": dataset_name,
        "guidance": guidance,
        "reference": reference,
        "guidance_path": str(guidance_path),
        "reference_path": str(reference_path),

        # Explicit statistical-guidance components
        "numeric_feature_guidance": guidance[
            "numeric_feature_guidance"
        ],
        "categorical_feature_guidance": guidance[
            "categorical_feature_guidance"
        ],
        "pearson_dependencies": guidance[
            "strongest_numeric_pearson_dependencies"
        ],
        "spearman_dependencies": guidance[
            "strongest_numeric_spearman_dependencies"
        ],
        "categorical_dependencies": guidance[
            "strongest_categorical_dependencies"
        ],

        # Reference statistical evidence
        "dataset_profile": reference[
            "dataset_profile"
        ],
        "feature_profiles": reference[
            "feature_profiles"
        ],
        "pearson_correlation": reference[
            "pearson_correlation"
        ],
        "spearman_correlation": reference[
            "spearman_correlation"
        ],
        "categorical_dependency": reference[
            "categorical_dependency"
        ],

        # Governance policies
        "target_policy": guidance[
            "target_policy"
        ],
        "identifier_policy": guidance[
            "identifier_policy"
        ],
        "provenance_policy": guidance[
            "provenance_policy"
        ],
        "evidence_policy": guidance[
            "evidence_policy"
        ],
    }

    print(
        f"✓ {dataset_name:<20} "
        f"guidance=loaded | "
        f"reference=loaded | "
        f"features={expected_generative_dimension}"
    )

# -----------------------------------------------------------------------------------------------
# Final statistical-profile validation
# -----------------------------------------------------------------------------------------------

if set(STATISTICAL_PROFILE.keys()) != set(EXPECTED_DATASETS):
    raise RuntimeError(
        "STATISTICAL_PROFILE does not contain exactly the expected datasets."
    )

for dataset_name in EXPECTED_DATASETS:

    profile = STATISTICAL_PROFILE[dataset_name]

    if not profile["numeric_feature_guidance"]:
        warnings.warn(
            f"No numeric feature guidance entries found for "
            f"'{dataset_name}'."
        )

    if not profile["categorical_feature_guidance"]:
        warnings.warn(
            f"No categorical feature guidance entries found for "
            f"'{dataset_name}'."
        )

print()
print("✓ Authoritative Notebook 03 statistical profiles loaded.")
print("✓ SPP-GAN statistical guidance registry initialized.")

4. LOAD STATISTICAL PROFILE
✓ Notebook 03 root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03
✓ adult_income         guidance=loaded | reference=loaded | features=15
✓ bank_marketing       guidance=loaded | reference=loaded | features=17
✓ diabetes_130us       guidance=loaded | reference=loaded | features=48

✓ Authoritative Notebook 03 statistical profiles loaded.
✓ SPP-GAN statistical guidance registry initialized.


In [6]:
# ==================================================================================================
# 5. DEFINE INPUT REPRESENTATION
# ==================================================================================================

print("=" * 100)
print("5. DEFINE INPUT REPRESENTATION")
print("=" * 100)


class SPPGANInputRepresentation:
    """
    Defines the logical input representation used by SPP-GAN.

    The representation separates:

    1. Generative data features
       - Numerical features
       - Categorical features

    2. Conditional information
       - Optional conditional vector used by the generator/discriminator

    3. Statistical guidance
       - External statistical reference information
       - Not treated as ordinary record-level features

    Raw identifiers and provenance columns are never part of
    the generative representation.
    """

    def __init__(
        self,
        num_numerical,
        num_categorical,
        categorical_cardinalities=None,
        conditional_dim=0,
        transformed_dim=None,
    ):
        self.num_numerical = int(num_numerical)
        self.num_categorical = int(num_categorical)

        self.categorical_cardinalities = (
            [int(v) for v in categorical_cardinalities]
            if categorical_cardinalities is not None
            else []
        )

        self.conditional_dim = int(conditional_dim)

        # The actual transformed dimension is determined by
        # the preprocessing/transformation implementation.
        self._transformed_dim = (
            int(transformed_dim)
            if transformed_dim is not None
            else None
        )

        if self.num_numerical < 0:
            raise ValueError(
                "num_numerical must be non-negative."
            )

        if self.num_categorical < 0:
            raise ValueError(
                "num_categorical must be non-negative."
            )

        if self.conditional_dim < 0:
            raise ValueError(
                "conditional_dim must be non-negative."
            )

        if len(self.categorical_cardinalities) != self.num_categorical:
            raise ValueError(
                "Number of categorical cardinalities must equal "
                "num_categorical."
            )

        if any(v <= 0 for v in self.categorical_cardinalities):
            raise ValueError(
                "Categorical cardinalities must be positive."
            )

        if self._transformed_dim is not None and self._transformed_dim <= 0:
            raise ValueError(
                "transformed_dim must be positive when provided."
            )

    @property
    def feature_dim(self):
        """
        Logical feature dimension before dataset-specific
        transformation expansion.
        """

        return (
            self.num_numerical
            + sum(self.categorical_cardinalities)
        )

    @property
    def transformed_dim(self):
        """
        Actual transformed tensor dimension.

        This value must be supplied by the authoritative
        preprocessing/transformation layer once the transformation
        has been fitted.
        """

        return self._transformed_dim

    @property
    def conditional_input_dim(self):
        """
        Dimension of the optional conditional vector.
        """

        return self.conditional_dim

    def set_transformed_dim(self, transformed_dim):
        """
        Register the actual transformed dimension produced by
        the fitted transformation layer.
        """

        transformed_dim = int(transformed_dim)

        if transformed_dim <= 0:
            raise ValueError(
                "transformed_dim must be positive."
            )

        self._transformed_dim = transformed_dim

    def summary(self):
        return {
            "num_numerical": self.num_numerical,
            "num_categorical": self.num_categorical,
            "categorical_cardinalities": (
                list(self.categorical_cardinalities)
            ),
            "conditional_dim": self.conditional_dim,
            "logical_feature_dim": self.feature_dim,
            "transformed_dim": self.transformed_dim,
        }


print("✓ SPP-GAN input representation interface defined.")
print("✓ Actual transformed dimension is delegated to the fitted transformation layer.")
print("✓ Conditional information is represented separately.")
print("✓ Statistical guidance remains external to the record-level feature tensor.")
print("✓ Raw identifiers/provenance are excluded.")

5. DEFINE INPUT REPRESENTATION
✓ SPP-GAN input representation interface defined.
✓ Actual transformed dimension is delegated to the fitted transformation layer.
✓ Conditional information is represented separately.
✓ Statistical guidance remains external to the record-level feature tensor.
✓ Raw identifiers/provenance are excluded.


In [7]:
# ==================================================================================================
# 6. DEFINE LATENT SPACE
# ==================================================================================================

print("=" * 100)
print("6. DEFINE LATENT SPACE")
print("=" * 100)

SPPGAN_LATENT_DIM = 128
SPPGAN_CONDITIONAL_DIM = 0

class SPPGANLatentSpace:
    """
    Latent-space definition for SPP-GAN.

    z ~ N(0, I)
    """

    def __init__(self, latent_dim=128):
        self.latent_dim = int(latent_dim)

    def sample(self, batch_size, device=None, dtype=torch.float32):
        return torch.randn(
            int(batch_size),
            self.latent_dim,
            device=device,
            dtype=dtype,
        )

    def summary(self):
        return {
            "latent_dim": self.latent_dim,
            "distribution": "standard_normal",
            "notation": "z ~ N(0, I)",
        }

LATENT_SPACE = SPPGANLatentSpace(
    latent_dim=SPPGAN_LATENT_DIM
)

print(f"✓ Latent dimension : {LATENT_SPACE.latent_dim}")
print("✓ Distribution     : Standard Normal")

6. DEFINE LATENT SPACE
✓ Latent dimension : 128
✓ Distribution     : Standard Normal


In [8]:
# ==================================================================================================
# 7. DEFINE GENERATOR
# ==================================================================================================

print("=" * 100)
print("7. DEFINE GENERATOR")
print("=" * 100)


class SPPGANGenerator(nn.Module):
    """
    SPP-GAN Generator.

    The generator maps a latent vector z and an optional conditional
    vector c into a differentiable tabular representation.

    Input:
        z ~ N(0, I)

        Optional condition:
        c

    Architecture:
        [z, c]
            ↓
        Linear
            ↓
        ReLU
            ↓
        Linear
            ↓
        ReLU
            ↓
        Shared representation
            ├── Numerical output head
            └── Categorical output heads

    Important:
        Categorical heads return logits. Differentiable categorical
        probabilities are produced later during the forward pass using
        Gumbel-Softmax when requested.

    Raw identifiers and provenance variables are never generated.
    """

    def __init__(
        self,
        latent_dim,
        hidden_dim_1,
        hidden_dim_2,
        num_numerical,
        categorical_cardinalities,
        conditional_dim=0,
    ):
        super().__init__()

        self.latent_dim = int(latent_dim)
        self.hidden_dim_1 = int(hidden_dim_1)
        self.hidden_dim_2 = int(hidden_dim_2)
        self.num_numerical = int(num_numerical)
        self.categorical_cardinalities = [
            int(v) for v in categorical_cardinalities
        ]
        self.conditional_dim = int(conditional_dim)

        if self.latent_dim <= 0:
            raise ValueError("latent_dim must be positive.")

        if self.hidden_dim_1 <= 0:
            raise ValueError("hidden_dim_1 must be positive.")

        if self.hidden_dim_2 <= 0:
            raise ValueError("hidden_dim_2 must be positive.")

        if self.num_numerical < 0:
            raise ValueError(
                "num_numerical must be non-negative."
            )

        if self.conditional_dim < 0:
            raise ValueError(
                "conditional_dim must be non-negative."
            )

        input_dim = self.latent_dim + self.conditional_dim

        # ------------------------------------------------------------------------------------------
        # Shared generator backbone
        # ------------------------------------------------------------------------------------------

        self.backbone = nn.Sequential(
            nn.Linear(input_dim, self.hidden_dim_1),
            nn.ReLU(),
            nn.Linear(self.hidden_dim_1, self.hidden_dim_2),
            nn.ReLU(),
        )

        # ------------------------------------------------------------------------------------------
        # Numerical output head
        # ------------------------------------------------------------------------------------------

        if self.num_numerical > 0:
            self.numerical_head = nn.Linear(
                self.hidden_dim_2,
                self.num_numerical,
            )
        else:
            self.numerical_head = None

        # ------------------------------------------------------------------------------------------
        # Categorical output heads
        # ------------------------------------------------------------------------------------------

        self.categorical_heads = nn.ModuleList(
            [
                nn.Linear(
                    self.hidden_dim_2,
                    cardinality,
                )
                for cardinality in self.categorical_cardinalities
            ]
        )

    def forward(
        self,
        z,
        condition=None,
        categorical_temperature=1.0,
        hard=False,
    ):
        """
        Forward pass.

        Parameters
        ----------
        z : torch.Tensor
            Latent vectors with shape [batch_size, latent_dim].

        condition : torch.Tensor or None
            Optional conditional vector with shape
            [batch_size, conditional_dim].

        categorical_temperature : float
            Temperature used by Gumbel-Softmax.

        hard : bool
            If True, uses hard one-hot outputs while preserving
            differentiability through the straight-through estimator.
        """

        if z.ndim != 2:
            raise ValueError(
                "z must be a 2D tensor [batch_size, latent_dim]."
            )

        if z.shape[1] != self.latent_dim:
            raise ValueError(
                f"Expected latent dimension {self.latent_dim}, "
                f"received {z.shape[1]}."
            )

        # ------------------------------------------------------------------------------------------
        # Conditional input
        # ------------------------------------------------------------------------------------------

        if self.conditional_dim > 0:

            if condition is None:
                raise ValueError(
                    "condition must be provided when conditional_dim > 0."
                )

            if condition.ndim != 2:
                raise ValueError(
                    "condition must be a 2D tensor "
                    "[batch_size, conditional_dim]."
                )

            if condition.shape[0] != z.shape[0]:
                raise ValueError(
                    "z and condition must have the same batch size."
                )

            if condition.shape[1] != self.conditional_dim:
                raise ValueError(
                    f"Expected conditional dimension "
                    f"{self.conditional_dim}, "
                    f"received {condition.shape[1]}."
                )

            generator_input = torch.cat(
                [z, condition],
                dim=1,
            )

        else:

            if condition is not None:
                raise ValueError(
                    "condition was supplied, but conditional_dim is 0."
                )

            generator_input = z

        # ------------------------------------------------------------------------------------------
        # Shared representation
        # ------------------------------------------------------------------------------------------

        h = self.backbone(generator_input)

        # ------------------------------------------------------------------------------------------
        # Numerical output
        # ------------------------------------------------------------------------------------------

        numerical = None

        if self.numerical_head is not None:
            numerical = self.numerical_head(h)

        # ------------------------------------------------------------------------------------------
        # Categorical outputs
        # ------------------------------------------------------------------------------------------

        categorical_logits = [
            head(h)
            for head in self.categorical_heads
        ]

        categorical_probabilities = []

        for logits in categorical_logits:

            probabilities = F.gumbel_softmax(
                logits,
                tau=float(categorical_temperature),
                hard=bool(hard),
                dim=1,
            )

            categorical_probabilities.append(
                probabilities
            )

        return {
            "hidden": h,
            "numerical": numerical,
            "categorical_logits": categorical_logits,
            "categorical_probabilities": categorical_probabilities,
        }

    def summary(self):
        return {
            "architecture": "MLP",
            "latent_dim": self.latent_dim,
            "conditional_dim": self.conditional_dim,
            "hidden_dim_1": self.hidden_dim_1,
            "hidden_dim_2": self.hidden_dim_2,
            "num_numerical": self.num_numerical,
            "num_categorical": len(
                self.categorical_cardinalities
            ),
            "categorical_cardinalities": list(
                self.categorical_cardinalities
            ),
        }


print("✓ SPP-GAN generator architecture defined.")
print("✓ Optional conditional input is supported explicitly.")
print("✓ Numerical and categorical output heads are separated.")
print("✓ Categorical outputs support differentiable Gumbel-Softmax.")
print("✓ Raw identifiers/provenance are excluded.")
print("✓ No training or synthetic-data generation performed.")

7. DEFINE GENERATOR
✓ SPP-GAN generator architecture defined.
✓ Optional conditional input is supported explicitly.
✓ Numerical and categorical output heads are separated.
✓ Categorical outputs support differentiable Gumbel-Softmax.
✓ Raw identifiers/provenance are excluded.
✓ No training or synthetic-data generation performed.


In [9]:
# ==================================================================================================
# 8. DEFINE DISCRIMINATOR / CRITIC
# ==================================================================================================

print("=" * 100)
print("8. DEFINE DISCRIMINATOR / CRITIC")
print("=" * 100)


class SPPGANCritic(nn.Module):
    """
    SPP-GAN scalar critic.

    Input:
        Differentiable transformed real or synthetic tabular
        representation.

    Output:
        One unrestricted scalar critic score per sample.

    The critic does NOT use:
        - sigmoid activation
        - binary cross-entropy
        - dropout
        - probability interpretation

    The critic is compatible with the WGAN-style adversarial
    objective used by SPP-GAN.
    """

    def __init__(
        self,
        input_dim,
        hidden_dim_1=256,
        hidden_dim_2=256,
    ):
        super().__init__()

        self.input_dim = int(input_dim)
        self.hidden_dim_1 = int(hidden_dim_1)
        self.hidden_dim_2 = int(hidden_dim_2)

        if self.input_dim <= 0:
            raise ValueError(
                "input_dim must be positive."
            )

        if self.hidden_dim_1 <= 0:
            raise ValueError(
                "hidden_dim_1 must be positive."
            )

        if self.hidden_dim_2 <= 0:
            raise ValueError(
                "hidden_dim_2 must be positive."
            )

        # ------------------------------------------------------------------------------------------
        # Scalar WGAN-style critic
        # ------------------------------------------------------------------------------------------

        self.network = nn.Sequential(
            nn.Linear(
                self.input_dim,
                self.hidden_dim_1,
            ),
            nn.LeakyReLU(0.2),

            nn.Linear(
                self.hidden_dim_1,
                self.hidden_dim_2,
            ),
            nn.LeakyReLU(0.2),

            nn.Linear(
                self.hidden_dim_2,
                1,
            ),
        )

    def forward(self, x):
        """
        Return one unrestricted scalar score per sample.

        Expected input shape:
            [batch_size, input_dim]

        Returned shape:
            [batch_size]
        """

        if x.ndim != 2:
            raise ValueError(
                "Critic input must be a 2D tensor "
                "[batch_size, input_dim]."
            )

        if x.shape[1] != self.input_dim:
            raise ValueError(
                f"Expected critic input dimension "
                f"{self.input_dim}, received {x.shape[1]}."
            )

        score = self.network(x)

        # Explicitly return one scalar score per sample.
        return score.squeeze(-1)

    def summary(self):
        return {
            "architecture": "MLP scalar WGAN-style critic",
            "input_dim": self.input_dim,
            "hidden_dim_1": self.hidden_dim_1,
            "hidden_dim_2": self.hidden_dim_2,
            "output_dim": 1,
            "output_type": "unrestricted_scalar_score",
            "sigmoid": False,
            "dropout": False,
        }


print("✓ Scalar WGAN-style critic architecture defined.")
print("✓ No sigmoid activation.")
print("✓ No binary classification output.")
print("✓ No dropout.")
print("✓ One unrestricted scalar score is produced per sample.")

8. DEFINE DISCRIMINATOR / CRITIC
✓ Scalar WGAN-style critic architecture defined.
✓ No sigmoid activation.
✓ No binary classification output.
✓ No dropout.
✓ One unrestricted scalar score is produced per sample.


In [10]:
# ==================================================================================================
# 9. DEFINE OUTPUT LAYERS
# ==================================================================================================

print("=" * 100)
print("9. DEFINE OUTPUT LAYERS")
print("=" * 100)

class SPPGANOutputLayers:
    """
    Unified output-layer specification.
    """

    def __init__(
        self,
        num_numerical,
        categorical_cardinalities,
    ):
        self.num_numerical = int(num_numerical)
        self.categorical_cardinalities = list(
            categorical_cardinalities
        )

    def summary(self):
        return {
            "numerical_output_dim": self.num_numerical,
            "categorical_output_dimensions": self.categorical_cardinalities,
            "categorical_head_count": len(
                self.categorical_cardinalities
            ),
        }

print("✓ Output-layer interface defined.")

9. DEFINE OUTPUT LAYERS
✓ Output-layer interface defined.


In [11]:
# ==================================================================================================
# 10. DEFINE NUMERICAL OUTPUT MECHANISM
# ==================================================================================================

print("=" * 100)
print("10. DEFINE NUMERICAL OUTPUT MECHANISM")
print("=" * 100)


class NumericalOutputMechanism:
    """
    Numerical output mechanism for SPP-GAN.

    The numerical output activation is explicitly configurable and
    must be consistent with the authoritative transformed representation
    established by Notebook 02.

    No independent normalization assumption is imposed at this stage.

    Dataset-specific inverse transformation remains delegated to the
    Notebook 02 preprocessing artifacts.
    """

    def __init__(
        self,
        activation="identity",
        output_range=None,
    ):
        self.activation = str(activation)

        if output_range is None:
            self.output_range = None
        else:
            if len(output_range) != 2:
                raise ValueError(
                    "output_range must contain exactly two values."
                )

            self.output_range = [
                float(output_range[0]),
                float(output_range[1]),
            ]

            if self.output_range[0] >= self.output_range[1]:
                raise ValueError(
                    "output_range lower bound must be smaller "
                    "than the upper bound."
                )

    def apply(self, x):
        """
        Apply the configured numerical output activation.

        The default identity mapping preserves the numerical output
        without imposing an artificial range.
        """

        if self.activation == "identity":
            return x

        if self.activation == "tanh":
            return torch.tanh(x)

        raise ValueError(
            f"Unsupported numerical activation: {self.activation}"
        )

    def summary(self):
        return {
            "type": "continuous",
            "activation": self.activation,
            "normalized_range": self.output_range,
            "inverse_transform": (
                "Notebook_02_preprocessing_artifacts"
            ),
        }


# -----------------------------------------------------------------------------------------------
# SPP-GAN numerical output interface
# -----------------------------------------------------------------------------------------------

NUMERICAL_OUTPUT = NumericalOutputMechanism(
    activation="identity",
    output_range=None,
)

print(
    f"✓ Numerical output activation : "
    f"{NUMERICAL_OUTPUT.activation}"
)

print(
    f"✓ Output range                : "
    f"{NUMERICAL_OUTPUT.output_range}"
)

print(
    "✓ Numerical representation is delegated to "
    "the authoritative Notebook 02 transformation."
)

print(
    "✓ Inverse transformation remains delegated to "
    "Notebook 02 preprocessing artifacts."
)

10. DEFINE NUMERICAL OUTPUT MECHANISM
✓ Numerical output activation : identity
✓ Output range                : None
✓ Numerical representation is delegated to the authoritative Notebook 02 transformation.
✓ Inverse transformation remains delegated to Notebook 02 preprocessing artifacts.


In [12]:
# ==================================================================================================
# 11. DEFINE CATEGORICAL OUTPUT MECHANISM
# ==================================================================================================

print("=" * 100)
print("11. DEFINE CATEGORICAL OUTPUT MECHANISM")
print("=" * 100)


class CategoricalOutputMechanism:
    """
    Categorical output mechanism for SPP-GAN.

    Generator categorical heads produce logits.

    During differentiable training:
        logits -> Gumbel-Softmax -> categorical probabilities

    During final synthetic-data generation:
        logits/probabilities -> hard category selection

    Hard decoding is therefore kept outside the differentiable
    generator-training path.
    """

    activation = "gumbel_softmax"

    @staticmethod
    def probabilities(
        logits,
        temperature=1.0,
        hard=False,
    ):
        """
        Produce differentiable categorical outputs.

        Parameters
        ----------
        logits : torch.Tensor
            Categorical logits with shape [batch_size, categories].

        temperature : float
            Gumbel-Softmax temperature.

        hard : bool
            If True, use hard one-hot values with the
            straight-through estimator.
        """

        if logits.ndim != 2:
            raise ValueError(
                "Categorical logits must be a 2D tensor "
                "[batch_size, num_categories]."
            )

        if float(temperature) <= 0:
            raise ValueError(
                "Gumbel-Softmax temperature must be positive."
            )

        return F.gumbel_softmax(
            logits,
            tau=float(temperature),
            hard=bool(hard),
            dim=-1,
        )

    @staticmethod
    def softmax_probabilities(logits):
        """
        Return ordinary categorical probabilities.

        This method is intended for inspection, evaluation,
        or decoding—not as the stochastic training sampler.
        """

        if logits.ndim != 2:
            raise ValueError(
                "Categorical logits must be a 2D tensor "
                "[batch_size, num_categories]."
            )

        return F.softmax(
            logits,
            dim=-1,
        )

    @staticmethod
    def sample(logits):
        """
        Perform non-differentiable hard categorical sampling.

        This method is intended for final synthetic-data generation
        and must not be used inside the generator optimization path.
        """

        if logits.ndim != 2:
            raise ValueError(
                "Categorical logits must be a 2D tensor "
                "[batch_size, num_categories]."
            )

        probabilities = F.softmax(
            logits,
            dim=-1,
        )

        return torch.multinomial(
            probabilities,
            num_samples=1,
        ).squeeze(-1)

    @classmethod
    def summary(cls):
        return {
            "type": "categorical",
            "training_activation": cls.activation,
            "training_representation": (
                "differentiable_gumbel_softmax"
            ),
            "probability_mapping": "softmax",
            "hard_sampling": "categorical_multinomial",
            "hard_decoding": "Notebook_13",
            "hard_sampling_during_training": False,
        }


CATEGORICAL_OUTPUT = CategoricalOutputMechanism()

print("✓ Categorical logits are converted through differentiable Gumbel-Softmax during training.")
print("✓ Softmax probabilities are available for inspection/evaluation.")
print("✓ Hard categorical sampling is isolated from the training path.")
print("✓ Hard decoding remains deferred to Notebook 13.")

11. DEFINE CATEGORICAL OUTPUT MECHANISM
✓ Categorical logits are converted through differentiable Gumbel-Softmax during training.
✓ Softmax probabilities are available for inspection/evaluation.
✓ Hard categorical sampling is isolated from the training path.
✓ Hard decoding remains deferred to Notebook 13.


In [13]:
# ==================================================================================================
# 12. DEFINE MODEL LOSS INTERFACE
# ==================================================================================================

print("=" * 100)
print("12. DEFINE MODEL LOSS INTERFACE")
print("=" * 100)


class SPPGANLossInterface:
    """
    Loss interface for downstream SPP-GAN training.

    Generator objective:

        L_G =
            L_adv
            + lambda_stat * L_stat

    where:

        L_adv  = -E[D(G(z))]

        L_stat =
            lambda_m * L_marg
            + lambda_mu * L_mom
            + lambda_d * L_dep
            + lambda_c * L_cat

    Differential privacy is NOT represented as an additive loss term.

    Instead, privacy is enforced through the DP training mechanism
    and evaluated/accounted for separately in the privacy-related
    notebooks.

    The actual statistical-guidance implementation is provided
    downstream by Notebook 09.
    """

    def __init__(
        self,
        lambda_stat=1.0,
    ):
        self.lambda_stat = float(lambda_stat)

        if self.lambda_stat < 0:
            raise ValueError(
                "lambda_stat must be non-negative."
            )

    def generator_loss(
        self,
        adversarial_loss,
        statistical_loss=0.0,
    ):
        """
        Compute the SPP-GAN generator objective.

        L_G = L_adv + lambda_stat * L_stat
        """

        return (
            adversarial_loss
            + self.lambda_stat * statistical_loss
        )

    @staticmethod
    def adversarial_generator_loss(
        fake_score,
    ):
        """
        WGAN-style generator adversarial objective.

        L_adv = -E[D(G(z))]
        """

        return -fake_score.mean()

    @staticmethod
    def critic_loss(
        real_score,
        fake_score,
    ):
        """
        WGAN-style critic objective.

        L_D =
            E[D(fake)] - E[D(real)]
        """

        return (
            fake_score.mean()
            - real_score.mean()
        )

    def summary(self):
        return {
            "generator_objective": (
                "L_G = L_adv + lambda_stat * L_stat"
            ),
            "adversarial_component": (
                "L_adv = -E[D(G(z))]"
            ),
            "statistical_component": (
                "Notebook_09"
            ),
            "privacy_mechanism": (
                "DP training and privacy accounting; "
                "not an additive loss"
            ),
            "lambda_stat": self.lambda_stat,
        }


LOSS_INTERFACE = SPPGANLossInterface(
    lambda_stat=1.0,
)

print("✓ SPP-GAN loss interface defined.")
print("✓ Generator objective: L_G = L_adv + lambda_stat × L_stat.")
print("✓ WGAN-style generator adversarial loss defined.")
print("✓ WGAN-style critic loss defined.")
print("✓ Privacy is treated as a DP constraint/mechanism, not an additive loss.")
print("✓ Statistical loss implementation is deferred to Notebook 09.")

12. DEFINE MODEL LOSS INTERFACE
✓ SPP-GAN loss interface defined.
✓ Generator objective: L_G = L_adv + lambda_stat × L_stat.
✓ WGAN-style generator adversarial loss defined.
✓ WGAN-style critic loss defined.
✓ Privacy is treated as a DP constraint/mechanism, not an additive loss.
✓ Statistical loss implementation is deferred to Notebook 09.


In [14]:
# ==================================================================================================
# 13. DEFINE STATISTICAL-GUIDANCE INTERFACE
# ==================================================================================================

print("=" * 100)
print("13. DEFINE STATISTICAL-GUIDANCE INTERFACE")
print("=" * 100)


class SPPGANStatisticalGuidance:
    """
    Statistical-guidance interface for SPP-GAN.

    Notebook 09 provides the actual differentiable implementation.

    The statistical objective is defined as:

        L_stat =
            lambda_m  * L_marg
            + lambda_mu * L_mom
            + lambda_d  * L_dep
            + lambda_c  * L_cat

    Components:

        L_marg
            Marginal distribution discrepancy.

        L_mom
            Numerical moment discrepancy.

        L_dep
            Numerical dependency/correlation discrepancy.

        L_cat
            Categorical dependency/distribution discrepancy.

    Statistical guidance is computed against the authoritative
    statistical reference established by Notebook 03.

    The statistical profile is a guidance/reference object and
    is not concatenated to the record-level generator input.

    No statistical guidance is enabled or trained in Notebook 08.
    """

    def __init__(
        self,
        lambda_m=1.0,
        lambda_mu=1.0,
        lambda_d=1.0,
        lambda_c=1.0,
    ):
        self.lambda_m = float(lambda_m)
        self.lambda_mu = float(lambda_mu)
        self.lambda_d = float(lambda_d)
        self.lambda_c = float(lambda_c)

        weights = {
            "lambda_m": self.lambda_m,
            "lambda_mu": self.lambda_mu,
            "lambda_d": self.lambda_d,
            "lambda_c": self.lambda_c,
        }

        if any(value < 0 for value in weights.values()):
            raise ValueError(
                "Statistical-guidance weights must be non-negative."
            )

        self.enabled = False
        self.method = "pending_notebook_09"

    def compute_loss(
        self,
        real_batch,
        synthetic_batch,
        statistical_reference=None,
    ):
        """
        Placeholder for the differentiable statistical-guidance loss.

        Notebook 09 supplies the actual implementation.

        The returned scalar must remain differentiable with respect
        to synthetic_batch so that gradients can propagate to the
        SPP-GAN generator during training.
        """

        if not isinstance(synthetic_batch, torch.Tensor):
            raise TypeError(
                "synthetic_batch must be a torch.Tensor."
            )

        return torch.zeros(
            (),
            device=synthetic_batch.device,
            dtype=synthetic_batch.dtype,
        )

    def summary(self):
        return {
            "enabled": self.enabled,
            "implementation": self.method,
            "training_notebook": "Notebook_09",
            "objective": (
                "L_stat = "
                "lambda_m*L_marg + "
                "lambda_mu*L_mom + "
                "lambda_d*L_dep + "
                "lambda_c*L_cat"
            ),
            "components": {
                "marginal": "L_marg",
                "moment": "L_mom",
                "dependency": "L_dep",
                "categorical": "L_cat",
            },
            "weights": {
                "lambda_m": self.lambda_m,
                "lambda_mu": self.lambda_mu,
                "lambda_d": self.lambda_d,
                "lambda_c": self.lambda_c,
            },
            "reference_source": "Notebook_03",
            "differentiable_requirement": True,
        }


STATISTICAL_GUIDANCE = SPPGANStatisticalGuidance(
    lambda_m=1.0,
    lambda_mu=1.0,
    lambda_d=1.0,
    lambda_c=1.0,
)

print("✓ Statistical-guidance interface defined.")
print("✓ Marginal, moment, dependency, and categorical components are explicitly registered.")
print("✓ Statistical reference is sourced from Notebook 03.")
print("✓ Statistical guidance remains external to the record-level input tensor.")
print("✓ Differentiable statistical loss is required for Notebook 09.")
print("✓ Implementation deferred to Notebook 09.")

13. DEFINE STATISTICAL-GUIDANCE INTERFACE
✓ Statistical-guidance interface defined.
✓ Marginal, moment, dependency, and categorical components are explicitly registered.
✓ Statistical reference is sourced from Notebook 03.
✓ Statistical guidance remains external to the record-level input tensor.
✓ Differentiable statistical loss is required for Notebook 09.
✓ Implementation deferred to Notebook 09.


In [15]:
# ==================================================================================================
# 14. DEFINE PRIVACY INTERFACE
# ==================================================================================================

print("=" * 100)
print("14. DEFINE PRIVACY INTERFACE")
print("=" * 100)

class SPPGANPrivacyInterface:
    """
    Privacy interface for SPP-GAN.

    Notebook 10:
        Differential privacy mechanism.

    Notebook 11:
        Privacy accounting and verification.

    No privacy budget is consumed in Notebook 08.
    """

    def __init__(
        self,
        enabled=False,
        epsilon=None,
        delta=None,
        max_grad_norm=None,
        accountant=None,
    ):
        self.enabled = bool(enabled)
        self.epsilon = epsilon
        self.delta = delta
        self.max_grad_norm = max_grad_norm
        self.accountant = accountant

    def attach(self, module):
        """
        Interface for attaching the future privacy mechanism.

        Actual DP attachment is implemented in Notebook 10.
        """
        return module

    def summary(self):
        return {
            "enabled": self.enabled,
            "epsilon": self.epsilon,
            "delta": self.delta,
            "max_grad_norm": self.max_grad_norm,
            "accountant": self.accountant,
            "implementation": "Notebook_10_and_11",
        }

PRIVACY_INTERFACE = SPPGANPrivacyInterface(
    enabled=False
)

print("✓ Privacy interface defined.")
print("✓ DP mechanism deferred to Notebook 10.")
print("✓ Privacy accounting deferred to Notebook 11.")

14. DEFINE PRIVACY INTERFACE
✓ Privacy interface defined.
✓ DP mechanism deferred to Notebook 10.
✓ Privacy accounting deferred to Notebook 11.


In [16]:
# ==================================================================================================
# 15. INITIALIZE SPP-GAN
# ==================================================================================================

print("=" * 100)
print("15. INITIALIZE SPP-GAN")
print("=" * 100)


# -----------------------------------------------------------------------------------------------
# 1. Imports required by Section 15
# -----------------------------------------------------------------------------------------------

import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


# -----------------------------------------------------------------------------------------------
# 2. Architecture configuration
# -----------------------------------------------------------------------------------------------

SPPGAN_CONFIG = {
    "latent_dim": 128,

    "generator": {
        "hidden_dim_1": 256,
        "hidden_dim_2": 256,
    },

    "critic": {
        "hidden_dim_1": 256,
        "hidden_dim_2": 256,
    },

    "loss": {
        "lambda_stat": 1.0,
        "objective": "L_G = L_adv + lambda_stat * L_stat",
    },

    "representation": {
        "numerical_activation": "identity",
        "categorical_training_activation": "gumbel_softmax",
        "categorical_probability_mapping": "softmax",
        "hard_decoding": "Notebook_13",
        "transformation_source": "Notebook_02",
    },

    "privacy": {
        "enabled": False,
        "implementation_notebook": 10,
        "accounting_notebook": 11,
    },

    "training": {
        "enabled": False,
        "implementation_notebook": 12,
    },

    "seed": MASTER_SEED,
}


# -----------------------------------------------------------------------------------------------
# 3. Notebook 02 authoritative metadata locations
# -----------------------------------------------------------------------------------------------

NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)

NB02_METADATA_ROOT = (
    NB02_ROOT
    / "schemas"
    / "metadata"
)


# -----------------------------------------------------------------------------------------------
# 4. Load authoritative Notebook 02 preprocessing metadata
# -----------------------------------------------------------------------------------------------

def load_notebook_02_preprocessing_metadata(dataset_name):
    """
    Load the authoritative dataset-specific Notebook 02
    preprocessing metadata.

    Feature types, generative columns, target policy, and
    transformed dimensions are inherited from Notebook 02.

    No feature types or transformed dimensions are inferred
    from sampled training data.
    """

    metadata_path = (
        NB02_METADATA_ROOT
        / f"{dataset_name}_preprocessing_metadata.json"
    )

    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Notebook 02 preprocessing metadata not found:\n"
            f"{metadata_path}"
        )

    with open(
        metadata_path,
        "r",
        encoding="utf-8",
    ) as f:
        metadata = json.load(f)

    if metadata.get("dataset_id") != dataset_name:
        raise ValueError(
            f"Dataset identity mismatch in Notebook 02 metadata: "
            f"expected {dataset_name!r}, "
            f"found {metadata.get('dataset_id')!r}."
        )

    required_keys = {
        "dataset_id",
        "training_rows",
        "input_columns",
        "numeric_columns",
        "categorical_columns",
        "target_column",
        "target_retained_in_training_dataset",
        "target_excluded_from_preprocessor_input",
        "identifier_columns",
        "provenance_column",
        "provenance_excluded_from_modeling",
        "identifiers_excluded_from_modeling",
        "generative_columns",
        "fit_dataset",
        "fit_scope",
        "fit_policy",
        "preprocessor_artifact",
        "schema_artifact",
        "preprocessing_feature_count",
        "generative_column_count",
        "target_retained_in_generative_schema",
        "target_excluded_from_transformed_features",
        "raw_target_manually_appended",
        "identifiers_excluded",
        "provenance_excluded_from_model_input",
        "transformed_feature_count",
        "transformed_feature_names",
    }

    missing_keys = required_keys.difference(
        metadata.keys()
    )

    if missing_keys:
        raise KeyError(
            f"Notebook 02 metadata for {dataset_name} is missing "
            f"required keys: {sorted(missing_keys)}"
        )

    return metadata


# -----------------------------------------------------------------------------------------------
# 5. Resolve exact categorical cardinalities
# -----------------------------------------------------------------------------------------------

def resolve_categorical_cardinalities(
    dataset_name,
    metadata,
):
    """
    Resolve categorical cardinalities from the fitted Notebook 02
    preprocessing artifact.

    Cardinalities are obtained from the fitted encoder rather than
    from a bounded sample of the training dataset.

    A cardinality of 1 is permitted because a categorical feature
    may contain only one observed training category.
    """

    categorical_columns = list(
        metadata["categorical_columns"]
    )

    if not categorical_columns:
        return []

    preprocessor_path = Path(
        metadata["preprocessor_artifact"]
    )

    if not preprocessor_path.exists():
        raise FileNotFoundError(
            f"Fitted preprocessor not found for {dataset_name}:\n"
            f"{preprocessor_path}"
        )

    fitted_preprocessor = joblib.load(
        preprocessor_path
    )

    # -------------------------------------------------------------------------------------------
    # Locate fitted categorical transformer
    # -------------------------------------------------------------------------------------------

    categorical_transformer = None

    if hasattr(
        fitted_preprocessor,
        "named_transformers_",
    ):
        named_transformers = (
            fitted_preprocessor.named_transformers_
        )

        for candidate_name in (
            "categorical",
            "cat",
            "categorical_transformer",
        ):
            if candidate_name in named_transformers:
                categorical_transformer = (
                    named_transformers[candidate_name]
                )
                break

    if categorical_transformer is None:
        raise RuntimeError(
            f"Unable to locate the fitted categorical transformer "
            f"for {dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Locate fitted categorical encoder
    # -------------------------------------------------------------------------------------------

    encoder = None

    if hasattr(
        categorical_transformer,
        "named_steps",
    ):
        named_steps = (
            categorical_transformer.named_steps
        )

        for step_name in (
            "onehot",
            "encoder",
            "categorical_encoder",
        ):
            if step_name in named_steps:
                candidate_encoder = (
                    named_steps[step_name]
                )

                if hasattr(
                    candidate_encoder,
                    "categories_",
                ):
                    encoder = candidate_encoder
                    break

    elif hasattr(
        categorical_transformer,
        "categories_",
    ):
        encoder = categorical_transformer

    if encoder is None:
        raise RuntimeError(
            f"Unable to locate a fitted categorical encoder "
            f"with categories_ for {dataset_name}."
        )

    fitted_categories = list(
        encoder.categories_
    )

    if len(fitted_categories) != len(
        categorical_columns
    ):
        raise ValueError(
            f"Categorical encoder dimension mismatch for "
            f"{dataset_name}: "
            f"{len(fitted_categories)} fitted categorical blocks "
            f"vs {len(categorical_columns)} categorical columns."
        )

    # -------------------------------------------------------------------------------------------
    # Extract cardinalities
    # -------------------------------------------------------------------------------------------

    cardinalities = []

    for column, categories in zip(
        categorical_columns,
        fitted_categories,
    ):
        cardinality = len(categories)

        if cardinality < 1:
            raise ValueError(
                f"Invalid categorical cardinality for "
                f"'{column}' in {dataset_name}: "
                f"{cardinality}"
            )

        if cardinality == 1:
            print(
                f"  ⚠ Single-category feature: "
                f"{column} → cardinality=1"
            )

        cardinalities.append(
            int(cardinality)
        )

    return cardinalities


# -----------------------------------------------------------------------------------------------
# 6. Initialize architecture registries
# -----------------------------------------------------------------------------------------------

SPPGAN_MODELS = {}
SPPGAN_ARCHITECTURES = {}


# -----------------------------------------------------------------------------------------------
# 7. Initialize one SPP-GAN architecture per dataset
# -----------------------------------------------------------------------------------------------

for dataset_name, schema in FEATURE_SCHEMAS.items():

    print("\n" + "-" * 100)
    print(
        f"Initializing SPP-GAN : {dataset_name}"
    )
    print("-" * 100)

    # -------------------------------------------------------------------------------------------
    # Load authoritative Notebook 02 metadata
    # -------------------------------------------------------------------------------------------

    metadata = (
        load_notebook_02_preprocessing_metadata(
            dataset_name
        )
    )

    numerical_columns = list(
        metadata["numeric_columns"]
    )

    categorical_columns = list(
        metadata["categorical_columns"]
    )

    generative_columns = list(
        metadata["generative_columns"]
    )

    target_column = metadata[
        "target_column"
    ]

    identifier_columns = list(
        metadata["identifier_columns"]
    )

    provenance_column = metadata[
        "provenance_column"
    ]

    transformed_dim = int(
        metadata["transformed_feature_count"]
    )

    transformed_feature_names = list(
        metadata["transformed_feature_names"]
    )

    # -------------------------------------------------------------------------------------------
    # Validate metadata against frozen Notebook 02 schema registry
    # -------------------------------------------------------------------------------------------

    if generative_columns != list(
        schema["generative_columns"]
    ):
        raise ValueError(
            f"Generative-column mismatch between Notebook 02 "
            f"metadata and Section 3 FEATURE_SCHEMAS for "
            f"{dataset_name}."
        )

    if target_column != schema["target"]:
        raise ValueError(
            f"Target mismatch for {dataset_name}: "
            f"Notebook 02 metadata={target_column!r}, "
            f"Section 3 schema={schema['target']!r}."
        )

    if len(generative_columns) != int(
        schema["generative_dimension"]
    ):
        raise ValueError(
            f"Generative dimension mismatch for {dataset_name}."
        )

    if len(numerical_columns) + len(
        categorical_columns
    ) != len(generative_columns) - 1:
        raise ValueError(
            f"Feature-type count mismatch for {dataset_name}. "
            f"Numerical + categorical feature count must equal "
            f"generative dimension minus target."
        )

    if target_column in numerical_columns:
        raise ValueError(
            f"Target column '{target_column}' incorrectly appears "
            f"in numerical features for {dataset_name}."
        )

    if target_column in categorical_columns:
        raise ValueError(
            f"Target column '{target_column}' incorrectly appears "
            f"in categorical features for {dataset_name}."
        )

    if provenance_column in numerical_columns:
        raise ValueError(
            f"Provenance column '{provenance_column}' appears "
            f"in numerical features for {dataset_name}."
        )

    if provenance_column in categorical_columns:
        raise ValueError(
            f"Provenance column '{provenance_column}' appears "
            f"in categorical features for {dataset_name}."
        )

    if set(identifier_columns) & set(
        numerical_columns + categorical_columns
    ):
        raise ValueError(
            f"Identifier columns leaked into model features "
            f"for {dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate target / identifier / provenance policy
    # -------------------------------------------------------------------------------------------

    if not metadata[
        "target_retained_in_generative_schema"
    ]:
        raise ValueError(
            f"Target retention policy invalid for {dataset_name}."
        )

    if not metadata[
        "target_excluded_from_preprocessor_input"
    ]:
        raise ValueError(
            f"Target must be excluded from preprocessor input "
            f"for {dataset_name}."
        )

    if not metadata[
        "target_excluded_from_transformed_features"
    ]:
        raise ValueError(
            f"Target must be excluded from transformed features "
            f"for {dataset_name}."
        )

    if not metadata[
        "provenance_excluded_from_modeling"
    ]:
        raise ValueError(
            f"Provenance column must be excluded from modeling "
            f"for {dataset_name}."
        )

    if not metadata[
        "provenance_excluded_from_model_input"
    ]:
        raise ValueError(
            f"Provenance column must be excluded from model input "
            f"for {dataset_name}."
        )

    if not metadata[
        "identifiers_excluded_from_modeling"
    ]:
        raise ValueError(
            f"Identifier exclusion policy invalid for {dataset_name}."
        )

    if not metadata[
        "identifiers_excluded"
    ]:
        raise ValueError(
            f"Identifier exclusion flag invalid for {dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate train-only preprocessing policy
    # -------------------------------------------------------------------------------------------

    if metadata["fit_policy"] != "train_only":
        raise ValueError(
            f"Notebook 02 preprocessing for {dataset_name} "
            f"is not marked train_only."
        )

    if metadata["fit_scope"] != "training_features_only":
        raise ValueError(
            f"Notebook 02 preprocessing for {dataset_name} "
            f"is not marked training_features_only."
        )

    if metadata["fit_dataset"] != "train_only":
        raise ValueError(
            f"Notebook 02 fit dataset policy invalid for "
            f"{dataset_name}: {metadata['fit_dataset']}"
        )

    # -------------------------------------------------------------------------------------------
    # Resolve exact categorical cardinalities from fitted encoder
    # -------------------------------------------------------------------------------------------

    categorical_cardinalities = (
        resolve_categorical_cardinalities(
            dataset_name,
            metadata,
        )
    )

    num_numerical = len(
        numerical_columns
    )

    num_categorical = len(
        categorical_columns
    )

    # -------------------------------------------------------------------------------------------
    # Validate transformed representation
    # -------------------------------------------------------------------------------------------

    if transformed_dim <= 0:
        raise ValueError(
            f"Invalid transformed dimension for "
            f"{dataset_name}: {transformed_dim}"
        )

    if len(transformed_feature_names) != (
        transformed_dim
    ):
        raise ValueError(
            f"Transformed feature-name count mismatch for "
            f"{dataset_name}: "
            f"count={len(transformed_feature_names)}, "
            f"declared={transformed_dim}."
        )

    # -------------------------------------------------------------------------------------------
    # Initialize input representation
    # -------------------------------------------------------------------------------------------

    input_representation = (
        SPPGANInputRepresentation(
            num_numerical=num_numerical,
            num_categorical=num_categorical,
            categorical_cardinalities=(
                categorical_cardinalities
            ),
            conditional_dim=SPPGAN_CONDITIONAL_DIM,
            transformed_dim=transformed_dim,
        )
    )

    # -------------------------------------------------------------------------------------------
    # Initialize generator
    # -------------------------------------------------------------------------------------------

    generator = SPPGANGenerator(
        latent_dim=SPPGAN_CONFIG[
            "latent_dim"
        ],
        hidden_dim_1=SPPGAN_CONFIG[
            "generator"
        ]["hidden_dim_1"],
        hidden_dim_2=SPPGAN_CONFIG[
            "generator"
        ]["hidden_dim_2"],
        num_numerical=num_numerical,
        categorical_cardinalities=(
            categorical_cardinalities
        ),
        conditional_dim=SPPGAN_CONDITIONAL_DIM,
    )

    # -------------------------------------------------------------------------------------------
    # Initialize scalar WGAN-style critic
    # -------------------------------------------------------------------------------------------

    critic = SPPGANCritic(
        input_dim=transformed_dim,
        hidden_dim_1=SPPGAN_CONFIG[
            "critic"
        ]["hidden_dim_1"],
        hidden_dim_2=SPPGAN_CONFIG[
            "critic"
        ]["hidden_dim_2"],
    )

    # -------------------------------------------------------------------------------------------
    # Store initialized models
    # -------------------------------------------------------------------------------------------

    SPPGAN_MODELS[dataset_name] = {
        "generator": generator,
        "critic": critic,
        "input_representation": (
            input_representation
        ),
    }

    # -------------------------------------------------------------------------------------------
    # Store architecture metadata
    # -------------------------------------------------------------------------------------------

    SPPGAN_ARCHITECTURES[dataset_name] = {
        "dataset": dataset_name,

        "target": target_column,

        "generative_columns": (
            generative_columns
        ),

        "identifier_columns": (
            identifier_columns
        ),

        "provenance_column": (
            provenance_column
        ),

        "numerical_columns": (
            numerical_columns
        ),

        "categorical_columns": (
            categorical_columns
        ),

        "categorical_cardinalities": (
            categorical_cardinalities
        ),

        "num_numerical": (
            num_numerical
        ),

        "num_categorical": (
            num_categorical
        ),

        "generative_dimension": (
            len(generative_columns)
        ),

        "transformed_dim": (
            transformed_dim
        ),

        "transformed_feature_names": (
            transformed_feature_names
        ),

        "latent_dim": (
            SPPGAN_CONFIG["latent_dim"]
        ),

        "conditional_dim": (
            SPPGAN_CONDITIONAL_DIM
        ),

        "generator_hidden_dims": [
            SPPGAN_CONFIG["generator"][
                "hidden_dim_1"
            ],
            SPPGAN_CONFIG["generator"][
                "hidden_dim_2"
            ],
        ],

        "critic_hidden_dims": [
            SPPGAN_CONFIG["critic"][
                "hidden_dim_1"
            ],
            SPPGAN_CONFIG["critic"][
                "hidden_dim_2"
            ],
        ],

        "representation": {
            "numerical_activation": (
                SPPGAN_CONFIG[
                    "representation"
                ]["numerical_activation"]
            ),
            "categorical_training_activation": (
                SPPGAN_CONFIG[
                    "representation"
                ]["categorical_training_activation"]
            ),
            "categorical_probability_mapping": (
                SPPGAN_CONFIG[
                    "representation"
                ]["categorical_probability_mapping"]
            ),
            "transformation_source": (
                "Notebook_02"
            ),
        },

        "loss": {
            "objective": (
                "L_G = L_adv + "
                "lambda_stat * L_stat"
            ),
            "lambda_stat": (
                SPPGAN_CONFIG[
                    "loss"
                ]["lambda_stat"]
            ),
        },

        "privacy": {
            "enabled": False,
            "implementation_notebook": 10,
            "accounting_notebook": 11,
        },

        "training": {
            "enabled": False,
            "implementation_notebook": 12,
        },
    }

    # -------------------------------------------------------------------------------------------
    # Dataset-level architecture summary
    # -------------------------------------------------------------------------------------------

    print(
        f"Numerical features   : "
        f"{num_numerical}"
    )

    print(
        f"Categorical features : "
        f"{num_categorical}"
    )

    print(
        f"Generative dimension : "
        f"{len(generative_columns)}"
    )

    print(
        f"Transformed dimension: "
        f"{transformed_dim}"
    )

    print(
        f"Latent dimension     : "
        f"{SPPGAN_CONFIG['latent_dim']}"
    )

    print(
        f"Target               : "
        f"{target_column}"
    )

    print(
        f"Identifiers excluded : "
        f"{len(identifier_columns)}"
    )


# -----------------------------------------------------------------------------------------------
# 8. Final architecture initialization status
# -----------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SPP-GAN ARCHITECTURE INITIALIZATION COMPLETE")
print("=" * 100)

print(
    f"✓ Datasets initialized : "
    f"{len(SPPGAN_MODELS)}"
)

print(
    f"✓ Dataset registry     : "
    f"{list(SPPGAN_MODELS.keys())}"
)

print(
    "✓ Notebook 02 remains the authoritative "
    "preprocessing source."
)

print(
    "✓ Feature types are inherited from "
    "Notebook 02 metadata."
)

print(
    "✓ Transformed dimensions are inherited "
    "from Notebook 02 metadata."
)

print(
    "✓ Categorical cardinalities are obtained "
    "from fitted preprocessing artifacts."
)

print(
    "✓ Single-category fitted features are "
    "retained without artificial category expansion."
)

print(
    "✓ Raw identifiers and provenance are "
    "excluded from model input."
)

print(
    "✓ Scalar WGAN-style critic initialized."
)

print(
    "✓ Statistical guidance remains external "
    "to the record-level feature tensor."
)

print(
    "✓ Privacy mechanism remains disabled "
    "in Notebook 08."
)

print(
    "✓ Training remains disabled in Notebook 08."
)

print(
    "✓ Synthetic generation remains disabled "
    "in Notebook 08."
)

print("=" * 100)

15. INITIALIZE SPP-GAN

----------------------------------------------------------------------------------------------------
Initializing SPP-GAN : adult_income
----------------------------------------------------------------------------------------------------
Numerical features   : 6
Categorical features : 8
Generative dimension : 15
Transformed dimension: 105
Latent dimension     : 128
Target               : income
Identifiers excluded : 0

----------------------------------------------------------------------------------------------------
Initializing SPP-GAN : bank_marketing
----------------------------------------------------------------------------------------------------
Numerical features   : 7
Categorical features : 9
Generative dimension : 17
Transformed dimension: 51
Latent dimension     : 128
Target               : y
Identifiers excluded : 0

----------------------------------------------------------------------------------------------------
Initializing SPP-GAN : diabetes

In [17]:
# ==================================================================================================
# 16. PARAMETER COUNT
# ==================================================================================================

print("=" * 100)
print("16. PARAMETER COUNT")
print("=" * 100)

def count_parameters(model):
    return int(
        sum(
            parameter.numel()
            for parameter in model.parameters()
            if parameter.requires_grad
        )
    )

PARAMETER_RECORDS = []

for dataset_name, models in SPPGAN_MODELS.items():

    generator_params = count_parameters(models["generator"])
    critic_params = count_parameters(models["critic"])
    total_params = generator_params + critic_params

    record = {
        "dataset": dataset_name,
        "generator_parameters": generator_params,
        "critic_parameters": critic_params,
        "total_trainable_parameters": total_params,
    }

    PARAMETER_RECORDS.append(record)

    print(
        f"{dataset_name:<20} "
        f"G={generator_params:,} | "
        f"C={critic_params:,} | "
        f"Total={total_params:,}"
    )

PARAMETER_COUNT_DF = pd.DataFrame(PARAMETER_RECORDS)

parameter_path = (
    DIRS["metadata"] / "sppgan_parameter_count.csv"
)

PARAMETER_COUNT_DF.to_csv(
    parameter_path,
    index=False,
)

print(f"\n✓ Saved : {parameter_path}")

16. PARAMETER COUNT
adult_income         G=125,801 | C=93,185 | Total=218,986
bank_marketing       G=111,923 | C=79,361 | Total=191,284
diabetes_130us       G=697,369 | C=662,529 | Total=1,359,898

✓ Saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/metadata/sppgan_parameter_count.csv


In [18]:
# ==================================================================================================
# 17. ARCHITECTURE SUMMARY
# ==================================================================================================

print("=" * 100)
print("17. ARCHITECTURE SUMMARY")
print("=" * 100)


# -----------------------------------------------------------------------------------------------
# 1. Initialize architecture summary registry
# -----------------------------------------------------------------------------------------------

ARCHITECTURE_SUMMARY = []


# -----------------------------------------------------------------------------------------------
# 2. Build dataset-level architecture summary
# -----------------------------------------------------------------------------------------------

for dataset_name, architecture in SPPGAN_ARCHITECTURES.items():

    models = SPPGAN_MODELS[dataset_name]

    generator_parameters = count_parameters(
        models["generator"]
    )

    critic_parameters = count_parameters(
        models["critic"]
    )

    total_parameters = (
        generator_parameters
        + critic_parameters
    )

    # -------------------------------------------------------------------------------------------
    # Authoritative dimensions
    # -------------------------------------------------------------------------------------------

    generative_dimension = int(
        architecture["generative_dimension"]
    )

    transformed_dimension = int(
        architecture["transformed_dim"]
    )

    # -------------------------------------------------------------------------------------------
    # Consistency validation
    # -------------------------------------------------------------------------------------------

    if generative_dimension <= 0:
        raise ValueError(
            f"Invalid generative dimension for {dataset_name}: "
            f"{generative_dimension}"
        )

    if transformed_dimension <= 0:
        raise ValueError(
            f"Invalid transformed dimension for {dataset_name}: "
            f"{transformed_dimension}"
        )

    if architecture["num_numerical"] < 0:
        raise ValueError(
            f"Invalid numerical feature count for {dataset_name}."
        )

    if architecture["num_categorical"] < 0:
        raise ValueError(
            f"Invalid categorical feature count for {dataset_name}."
        )

    if (
        architecture["num_numerical"]
        + architecture["num_categorical"]
        != generative_dimension - 1
    ):
        raise ValueError(
            f"Feature-count inconsistency for {dataset_name}: "
            f"numerical + categorical must equal "
            f"generative dimension - 1."
        )

    # -------------------------------------------------------------------------------------------
    # Build summary record
    # -------------------------------------------------------------------------------------------

    summary = {
        "dataset": dataset_name,

        "target": architecture[
            "target"
        ],

        "generative_dimension": (
            generative_dimension
        ),

        "transformed_dimension": (
            transformed_dimension
        ),

        "numerical_features": (
            architecture[
                "num_numerical"
            ]
        ),

        "categorical_features": (
            architecture[
                "num_categorical"
            ]
        ),

        "latent_dim": architecture[
            "latent_dim"
        ],

        "generator_hidden_1": (
            architecture[
                "generator_hidden_dims"
            ][0]
        ),

        "generator_hidden_2": (
            architecture[
                "generator_hidden_dims"
            ][1]
        ),

        "critic_hidden_1": (
            architecture[
                "critic_hidden_dims"
            ][0]
        ),

        "critic_hidden_2": (
            architecture[
                "critic_hidden_dims"
            ][1]
        ),

        "generator_parameters": (
            generator_parameters
        ),

        "critic_parameters": (
            critic_parameters
        ),

        "total_trainable_parameters": (
            total_parameters
        ),

        # ---------------------------------------------------------------------------------------
        # Representation
        # ---------------------------------------------------------------------------------------

        "numerical_activation": (
            architecture[
                "representation"
            ][
                "numerical_activation"
            ]
        ),

        "categorical_training_activation": (
            architecture[
                "representation"
            ][
                "categorical_training_activation"
            ]
        ),

        "categorical_probability_mapping": (
            architecture[
                "representation"
            ][
                "categorical_probability_mapping"
            ]
        ),

        "hard_decoding": (
            architecture[
                "representation"
            ][
                "hard_decoding"
            ]
        ) if "hard_decoding" in architecture[
            "representation"
        ] else "Notebook_13",

        "critic_output": "scalar",

        # ---------------------------------------------------------------------------------------
        # SPP-GAN methodological components
        # ---------------------------------------------------------------------------------------

        "statistical_guidance": (
            "Notebook_09"
        ),

        "differential_privacy": (
            "Notebook_10"
        ),

        "privacy_accounting": (
            "Notebook_11"
        ),

        "training": (
            "Notebook_12"
        ),
    }

    ARCHITECTURE_SUMMARY.append(
        summary
    )


# -----------------------------------------------------------------------------------------------
# 3. Convert to DataFrame
# -----------------------------------------------------------------------------------------------

ARCHITECTURE_SUMMARY_DF = pd.DataFrame(
    ARCHITECTURE_SUMMARY
)


# -----------------------------------------------------------------------------------------------
# 4. Validate final architecture summary
# -----------------------------------------------------------------------------------------------

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

actual_datasets = (
    ARCHITECTURE_SUMMARY_DF[
        "dataset"
    ].tolist()
)

if actual_datasets != EXPECTED_DATASETS:
    raise ValueError(
        "Architecture summary dataset order/content mismatch.\n"
        f"Expected: {EXPECTED_DATASETS}\n"
        f"Found   : {actual_datasets}"
    )


# -----------------------------------------------------------------------------------------------
# 5. Validate authoritative representation settings
# -----------------------------------------------------------------------------------------------

if not all(
    ARCHITECTURE_SUMMARY_DF[
        "numerical_activation"
    ] == "identity"
):
    raise ValueError(
        "Numerical activation mismatch. "
        "Section 10 defines the numerical output mechanism "
        "as identity."
    )


if not all(
    ARCHITECTURE_SUMMARY_DF[
        "categorical_training_activation"
    ] == "gumbel_softmax"
):
    raise ValueError(
        "Categorical training activation mismatch. "
        "Section 11 defines differentiable Gumbel-Softmax "
        "for categorical training."
    )


if not all(
    ARCHITECTURE_SUMMARY_DF[
        "categorical_probability_mapping"
    ] == "softmax"
):
    raise ValueError(
        "Categorical probability mapping mismatch. "
        "Section 11 defines softmax probabilities for "
        "inspection/evaluation."
    )


if not all(
    ARCHITECTURE_SUMMARY_DF[
        "critic_output"
    ] == "scalar"
):
    raise ValueError(
        "Critic output mismatch. "
        "Section 8 defines a scalar WGAN-style critic."
    )


# -----------------------------------------------------------------------------------------------
# 6. Save architecture summary
# -----------------------------------------------------------------------------------------------

summary_path = (
    DIRS["architecture"]
    / "sppgan_architecture_summary.csv"
)

ARCHITECTURE_SUMMARY_DF.to_csv(
    summary_path,
    index=False,
)


# -----------------------------------------------------------------------------------------------
# 7. Display final architecture summary
# -----------------------------------------------------------------------------------------------

print(
    ARCHITECTURE_SUMMARY_DF.to_string(
        index=False
    )
)

print(
    f"\n✓ Saved : {summary_path}"
)

print(
    "\n✓ Generative dimension and transformed dimension "
    "are reported separately."
)

print(
    "✓ Numerical activation correctly recorded as identity."
)

print(
    "✓ Gumbel-Softmax correctly recorded as the "
    "categorical training mechanism."
)

print(
    "✓ Softmax correctly recorded as the categorical "
    "probability mapping."
)

print(
    "✓ Scalar WGAN-style critic correctly recorded."
)

print(
    "✓ Statistical guidance, privacy, accounting, and "
    "training are correctly delegated to subsequent notebooks."
)

17. ARCHITECTURE SUMMARY
       dataset     target  generative_dimension  transformed_dimension  numerical_features  categorical_features  latent_dim  generator_hidden_1  generator_hidden_2  critic_hidden_1  critic_hidden_2  generator_parameters  critic_parameters  total_trainable_parameters numerical_activation categorical_training_activation categorical_probability_mapping hard_decoding critic_output statistical_guidance differential_privacy privacy_accounting    training
  adult_income     income                    15                    105                   6                     8         128                 256                 256              256              256                125801              93185                      218986             identity                  gumbel_softmax                         softmax   Notebook_13        scalar          Notebook_09          Notebook_10        Notebook_11 Notebook_12
bank_marketing          y                    17                    

In [19]:
# ==================================================================================================
# 18. TENSOR SHAPE TESTS
# ==================================================================================================

print("=" * 100)
print("18. TENSOR SHAPE TESTS")
print("=" * 100)

SHAPE_TEST_RESULTS = []

TEST_BATCH_SIZE = 4

for dataset_name, architecture in SPPGAN_ARCHITECTURES.items():

    generator = SPPGAN_MODELS[dataset_name]["generator"]

    latent_dim = architecture["latent_dim"]
    transformed_dim = architecture["transformed_dim"]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    generator = generator.to(device)

    z = torch.randn(
        TEST_BATCH_SIZE,
        latent_dim,
        device=device,
    )

    outputs = generator(z)

    passed = True

    expected_numerical = architecture["num_numerical"]

    if expected_numerical > 0:

        actual_shape = tuple(outputs["numerical"].shape)

        expected_shape = (
            TEST_BATCH_SIZE,
            expected_numerical,
        )

        passed &= actual_shape == expected_shape

    for logits, cardinality in zip(
        outputs["categorical_logits"],
        architecture["categorical_cardinalities"],
    ):

        actual_shape = tuple(logits.shape)

        expected_shape = (
            TEST_BATCH_SIZE,
            cardinality,
        )

        passed &= actual_shape == expected_shape

    record = {
        "dataset": dataset_name,
        "batch_size": TEST_BATCH_SIZE,
        "latent_dim": latent_dim,
        "transformed_dim": transformed_dim,
        "shape_test": "PASS" if passed else "FAIL",
    }

    SHAPE_TEST_RESULTS.append(record)

    print(
        f"{dataset_name:<20} "
        f"shape test : {'PASS' if passed else 'FAIL'}"
    )

    if not passed:
        raise AssertionError(
            f"Tensor shape test failed for {dataset_name}."
        )

SHAPE_TEST_DF = pd.DataFrame(SHAPE_TEST_RESULTS)

shape_path = (
    DIRS["validation"] / "sppgan_tensor_shape_tests.csv"
)

SHAPE_TEST_DF.to_csv(
    shape_path,
    index=False,
)

print(f"\n✓ Saved : {shape_path}")

18. TENSOR SHAPE TESTS
adult_income         shape test : PASS
bank_marketing       shape test : PASS
diabetes_130us       shape test : PASS

✓ Saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/validation/sppgan_tensor_shape_tests.csv


In [20]:
# ==================================================================================================
# 19. FORWARD-PASS TEST
# ==================================================================================================

print("=" * 100)
print("19. FORWARD-PASS TEST")
print("=" * 100)


# -----------------------------------------------------------------------------------------------
# 1. Initialize results
# -----------------------------------------------------------------------------------------------

FORWARD_TEST_RESULTS = []

TEST_BATCH_SIZE = 4


# -----------------------------------------------------------------------------------------------
# 2. Forward-pass validation
# -----------------------------------------------------------------------------------------------

for dataset_name, architecture in SPPGAN_ARCHITECTURES.items():

    models = SPPGAN_MODELS[dataset_name]

    generator = models["generator"]
    critic = models["critic"]

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    generator = generator.to(device)
    critic = critic.to(device)

    generator.eval()
    critic.eval()

    # -------------------------------------------------------------------------------------------
    # Generate latent input
    # -------------------------------------------------------------------------------------------

    z = torch.randn(
        TEST_BATCH_SIZE,
        architecture["latent_dim"],
        device=device,
    )

    # -------------------------------------------------------------------------------------------
    # Forward pass
    # -------------------------------------------------------------------------------------------

    with torch.no_grad():

        generated = generator(z)

        # ---------------------------------------------------------------------------------------
        # Reconstruct transformed representation
        # ---------------------------------------------------------------------------------------

        pieces = []

        if generated["numerical"] is not None:
            pieces.append(
                generated["numerical"]
            )

        for logits in generated[
            "categorical_logits"
        ]:
            pieces.append(
                F.softmax(
                    logits,
                    dim=-1,
                )
            )

        if not pieces:
            raise RuntimeError(
                f"No generator outputs found for "
                f"{dataset_name}."
            )

        synthetic_representation = torch.cat(
            pieces,
            dim=1,
        )

        # ---------------------------------------------------------------------------------------
        # Critic forward pass
        # ---------------------------------------------------------------------------------------

        critic_score = critic(
            synthetic_representation
        )

    # -------------------------------------------------------------------------------------------
    # Expected dimensions
    # -------------------------------------------------------------------------------------------

    expected_dim = int(
        architecture["transformed_dim"]
    )

    # -------------------------------------------------------------------------------------------
    # Validate generator representation dimension
    # -------------------------------------------------------------------------------------------

    if synthetic_representation.ndim != 2:
        raise AssertionError(
            f"Generator representation must be 2-dimensional "
            f"for {dataset_name}. "
            f"Observed shape: "
            f"{tuple(synthetic_representation.shape)}"
        )

    if synthetic_representation.shape != (
        TEST_BATCH_SIZE,
        expected_dim,
    ):
        raise AssertionError(
            f"Forward representation dimension mismatch "
            f"for {dataset_name}: "
            f"{tuple(synthetic_representation.shape)} != "
            f"{(TEST_BATCH_SIZE, expected_dim)}"
        )

    # -------------------------------------------------------------------------------------------
    # Validate scalar WGAN critic output
    # -------------------------------------------------------------------------------------------
    #
    # Section 8 defines the critic as producing one unrestricted
    # scalar score per sample. Therefore the expected tensor
    # shape is [batch_size], not [batch_size, 1].
    # -------------------------------------------------------------------------------------------

    if critic_score.ndim != 1:
        raise AssertionError(
            f"Critic must return one scalar score per sample "
            f"for {dataset_name}. "
            f"Expected shape: "
            f"({TEST_BATCH_SIZE},), "
            f"observed: "
            f"{tuple(critic_score.shape)}"
        )

    if critic_score.shape != (
        TEST_BATCH_SIZE,
    ):
        raise AssertionError(
            f"Critic output shape mismatch for "
            f"{dataset_name}: "
            f"{tuple(critic_score.shape)} != "
            f"{(TEST_BATCH_SIZE,)}"
        )

    # -------------------------------------------------------------------------------------------
    # Validate finite generator output
    # -------------------------------------------------------------------------------------------

    if not torch.isfinite(
        synthetic_representation
    ).all():
        raise AssertionError(
            f"Non-finite generator output for "
            f"{dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate finite critic output
    # -------------------------------------------------------------------------------------------

    if not torch.isfinite(
        critic_score
    ).all():
        raise AssertionError(
            f"Non-finite critic output for "
            f"{dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Record successful test
    # -------------------------------------------------------------------------------------------

    FORWARD_TEST_RESULTS.append({
        "dataset": dataset_name,

        "batch_size": TEST_BATCH_SIZE,

        "latent_dim": architecture[
            "latent_dim"
        ],

        "generated_shape": str(
            tuple(
                synthetic_representation.shape
            )
        ),

        "expected_transformed_dim": (
            expected_dim
        ),

        "critic_shape": str(
            tuple(
                critic_score.shape
            )
        ),

        "critic_output_type": (
            "scalar_per_sample"
        ),

        "finite_output": True,

        "forward_test": "PASS",
    })

    print(
        f"{dataset_name:<20} "
        f"generated="
        f"{tuple(synthetic_representation.shape)} | "
        f"critic="
        f"{tuple(critic_score.shape)} | "
        f"PASS"
    )


# -----------------------------------------------------------------------------------------------
# 3. Create validation DataFrame
# -----------------------------------------------------------------------------------------------

FORWARD_TEST_DF = pd.DataFrame(
    FORWARD_TEST_RESULTS
)


# -----------------------------------------------------------------------------------------------
# 4. Final validation
# -----------------------------------------------------------------------------------------------

if len(FORWARD_TEST_DF) != len(
    SPPGAN_ARCHITECTURES
):
    raise AssertionError(
        "Forward-pass test did not produce one "
        "result per dataset."
    )

if not all(
    FORWARD_TEST_DF[
        "forward_test"
    ] == "PASS"
):
    raise AssertionError(
        "One or more forward-pass tests failed."
    )


# -----------------------------------------------------------------------------------------------
# 5. Save validation artifact
# -----------------------------------------------------------------------------------------------

forward_path = (
    DIRS["validation"]
    / "sppgan_forward_pass_tests.csv"
)

FORWARD_TEST_DF.to_csv(
    forward_path,
    index=False,
)


# -----------------------------------------------------------------------------------------------
# 6. Final status
# -----------------------------------------------------------------------------------------------

print(
    f"\n✓ Saved : {forward_path}"
)

print(
    "✓ Generator forward pass validated."
)

print(
    "✓ Transformed representation dimensions validated."
)

print(
    "✓ Scalar WGAN-style critic output validated."
)

print(
    "✓ One critic score is produced per sample."
)

print(
    "✓ All forward-pass outputs are finite."
)

19. FORWARD-PASS TEST
adult_income         generated=(4, 105) | critic=(4,) | PASS
bank_marketing       generated=(4, 51) | critic=(4,) | PASS
diabetes_130us       generated=(4, 2329) | critic=(4,) | PASS

✓ Saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/validation/sppgan_forward_pass_tests.csv
✓ Generator forward pass validated.
✓ Transformed representation dimensions validated.
✓ Scalar WGAN-style critic output validated.
✓ One critic score is produced per sample.
✓ All forward-pass outputs are finite.


In [21]:
# ==================================================================================================
# 20. BACKWARD-PASS TEST
# ==================================================================================================

print("=" * 100)
print("20. BACKWARD-PASS TEST")
print("=" * 100)

BACKWARD_TEST_RESULTS = []

for dataset_name, architecture in SPPGAN_ARCHITECTURES.items():

    models = SPPGAN_MODELS[dataset_name]

    generator = models["generator"]
    critic = models["critic"]

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    generator = generator.to(device)
    critic = critic.to(device)

    generator.train()
    critic.train()

    generator.zero_grad(set_to_none=True)
    critic.zero_grad(set_to_none=True)

    z = torch.randn(
        TEST_BATCH_SIZE,
        architecture["latent_dim"],
        device=device,
        requires_grad=True,
    )

    generated = generator(z)

    pieces = []

    if generated["numerical"] is not None:
        pieces.append(generated["numerical"])

    for logits in generated["categorical_logits"]:
        pieces.append(
            F.softmax(logits, dim=-1)
        )

    synthetic_representation = torch.cat(
        pieces,
        dim=1,
    )

    critic_score = critic(
        synthetic_representation
    )

    loss = -critic_score.mean()

    if not torch.isfinite(loss):
        raise AssertionError(
            f"Non-finite backward loss for {dataset_name}."
        )

    loss.backward()

    BACKWARD_TEST_RESULTS.append({
        "dataset": dataset_name,
        "loss": float(loss.detach().cpu()),
        "backward_test": "PASS",
    })

    print(
        f"{dataset_name:<20} "
        f"loss={float(loss.detach().cpu()):.6f} | PASS"
    )

BACKWARD_TEST_DF = pd.DataFrame(
    BACKWARD_TEST_RESULTS
)

backward_path = (
    DIRS["validation"]
    / "sppgan_backward_pass_tests.csv"
)

BACKWARD_TEST_DF.to_csv(
    backward_path,
    index=False,
)

print(f"\n✓ Saved : {backward_path}")

20. BACKWARD-PASS TEST
adult_income         loss=-0.014414 | PASS
bank_marketing       loss=0.013476 | PASS
diabetes_130us       loss=-0.038668 | PASS

✓ Saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/validation/sppgan_backward_pass_tests.csv


In [22]:
# ==================================================================================================
# 21. GRADIENT VALIDATION
# ==================================================================================================

print("=" * 100)
print("21. GRADIENT VALIDATION")
print("=" * 100)

GRADIENT_RESULTS = []

for dataset_name, architecture in SPPGAN_ARCHITECTURES.items():

    models = SPPGAN_MODELS[dataset_name]

    generator = models["generator"]
    critic = models["critic"]

    generator.zero_grad(set_to_none=True)
    critic.zero_grad(set_to_none=True)

    device = next(generator.parameters()).device

    z = torch.randn(
        TEST_BATCH_SIZE,
        architecture["latent_dim"],
        device=device,
    )

    generated = generator(z)

    pieces = []

    if generated["numerical"] is not None:
        pieces.append(generated["numerical"])

    for logits in generated["categorical_logits"]:
        pieces.append(
            F.softmax(logits, dim=-1)
        )

    synthetic_representation = torch.cat(
        pieces,
        dim=1,
    )

    critic_score = critic(
        synthetic_representation
    )

    loss = -critic_score.mean()
    loss.backward()

    generator_gradients = []
    critic_gradients = []

    for parameter in generator.parameters():

        if parameter.grad is not None:
            generator_gradients.append(
                parameter.grad.detach()
            )

    for parameter in critic.parameters():

        if parameter.grad is not None:
            critic_gradients.append(
                parameter.grad.detach()
            )

    if not generator_gradients:
        raise AssertionError(
            f"No generator gradients for {dataset_name}."
        )

    if not critic_gradients:
        raise AssertionError(
            f"No critic gradients for {dataset_name}."
        )

    generator_finite = all(
        torch.isfinite(g).all().item()
        for g in generator_gradients
    )

    critic_finite = all(
        torch.isfinite(g).all().item()
        for g in critic_gradients
    )

    if not generator_finite:
        raise AssertionError(
            f"Non-finite generator gradients for {dataset_name}."
        )

    if not critic_finite:
        raise AssertionError(
            f"Non-finite critic gradients for {dataset_name}."
        )

    generator_norm = math.sqrt(
        sum(
            float(g.pow(2).sum().cpu())
            for g in generator_gradients
        )
    )

    critic_norm = math.sqrt(
        sum(
            float(g.pow(2).sum().cpu())
            for g in critic_gradients
        )
    )

    GRADIENT_RESULTS.append({
        "dataset": dataset_name,
        "generator_gradient_norm": generator_norm,
        "critic_gradient_norm": critic_norm,
        "generator_gradients_finite": generator_finite,
        "critic_gradients_finite": critic_finite,
        "gradient_test": "PASS",
    })

    print(
        f"{dataset_name:<20} "
        f"G-norm={generator_norm:.6f} | "
        f"C-norm={critic_norm:.6f} | PASS"
    )

GRADIENT_VALIDATION_DF = pd.DataFrame(
    GRADIENT_RESULTS
)

gradient_path = (
    DIRS["validation"]
    / "sppgan_gradient_validation.csv"
)

GRADIENT_VALIDATION_DF.to_csv(
    gradient_path,
    index=False,
)

print(f"\n✓ Saved : {gradient_path}")

21. GRADIENT VALIDATION
adult_income         G-norm=0.087788 | C-norm=1.326264 | PASS
bank_marketing       G-norm=0.093881 | C-norm=1.621959 | PASS
diabetes_130us       G-norm=0.022692 | C-norm=1.317673 | PASS

✓ Saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/validation/sppgan_gradient_validation.csv


In [23]:
# ==================================================================================================
# 22. SAVE ARCHITECTURE
# ==================================================================================================

print("=" * 100)
print("22. SAVE ARCHITECTURE")
print("=" * 100)

ARCHITECTURE_ARTIFACTS = []

for dataset_name, architecture in SPPGAN_ARCHITECTURES.items():

    models = SPPGAN_MODELS[dataset_name]

    dataset_model_dir = (
        DIRS["models"] / dataset_name
    )

    dataset_model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    generator_path = (
        dataset_model_dir
        / "sppgan_generator_state.pt"
    )

    critic_path = (
        dataset_model_dir
        / "sppgan_critic_state.pt"
    )

    architecture_path = (
        dataset_model_dir
        / "sppgan_architecture.json"
    )

    # -------------------------------------------------------------------------------------------
    # State dictionaries
    # -------------------------------------------------------------------------------------------

    torch.save(
        models["generator"].state_dict(),
        generator_path,
    )

    torch.save(
        models["critic"].state_dict(),
        critic_path,
    )

    # -------------------------------------------------------------------------------------------
    # Architecture metadata
    # -------------------------------------------------------------------------------------------

    architecture_metadata = {
        **architecture,
        "framework": FRAMEWORK_NAME,
        "notebook": NOTEBOOK_ID,
        "research_title": RESEARCH_TITLE,
        "config": SPPGAN_CONFIG,
        "input_representation": SPPGANInputRepresentation(
            architecture["num_numerical"],
            architecture["num_categorical"],
            architecture["categorical_cardinalities"],
        ).summary(),
        "latent_space": LATENT_SPACE.summary(),
        "numerical_output": NUMERICAL_OUTPUT.summary(),
        "categorical_output": CATEGORICAL_OUTPUT.summary(),
        "loss_interface": LOSS_INTERFACE.summary(),
        "statistical_guidance": STATISTICAL_GUIDANCE.summary(),
        "privacy_interface": PRIVACY_INTERFACE.summary(),
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    with open(
        architecture_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            architecture_metadata,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    # -------------------------------------------------------------------------------------------
    # Hash artifacts
    # -------------------------------------------------------------------------------------------

    def sha256_file(path):

        h = hashlib.sha256()

        with open(path, "rb") as f:

            for chunk in iter(
                lambda: f.read(1024 * 1024),
                b"",
            ):
                h.update(chunk)

        return h.hexdigest()

    ARCHITECTURE_ARTIFACTS.append({
        "dataset": dataset_name,
        "generator_state_path": str(generator_path),
        "generator_sha256": sha256_file(generator_path),
        "critic_state_path": str(critic_path),
        "critic_sha256": sha256_file(critic_path),
        "architecture_json_path": str(architecture_path),
        "architecture_json_sha256": sha256_file(
            architecture_path
        ),
    })

    print(f"\n✓ {dataset_name}")
    print(f"  Generator : {generator_path}")
    print(f"  Critic    : {critic_path}")
    print(f"  Metadata  : {architecture_path}")

ARCHITECTURE_ARTIFACTS_DF = pd.DataFrame(
    ARCHITECTURE_ARTIFACTS
)

artifact_path = (
    DIRS["metadata"]
    / "sppgan_architecture_artifacts.csv"
)

ARCHITECTURE_ARTIFACTS_DF.to_csv(
    artifact_path,
    index=False,
)

print(f"\n✓ Artifact registry saved : {artifact_path}")

22. SAVE ARCHITECTURE

✓ adult_income
  Generator : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/models/adult_income/sppgan_generator_state.pt
  Critic    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/models/adult_income/sppgan_critic_state.pt
  Metadata  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/models/adult_income/sppgan_architecture.json

✓ bank_marketing
  Generator : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/models/bank_marketing/sppgan_generator_state.pt
  Critic    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/models/bank_marketing/sppgan_critic_state.pt
  Metadata  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/models/bank_marketing/sppgan_architecture.json

✓ diabetes_130us
  Generator : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/models/diabetes_130us/sppgan_generator_state.pt
  Critic    : /c

In [24]:
# ==================================================================================================
# 23. SAVE MODEL CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("23. SAVE MODEL CONFIGURATION")
print("=" * 100)

MODEL_CONFIGURATION = {
    "framework": FRAMEWORK_NAME,
    "notebook": NOTEBOOK_ID,
    "research_title": RESEARCH_TITLE,
    "project_root": str(PROJECT_ROOT),

    "master_seed": MASTER_SEED,

    "training_enabled": False,
    "privacy_enabled": False,

    "architecture": SPPGAN_CONFIG,

    "datasets": {
        dataset_name: {
            "target": architecture["target"],
            "generative_columns": architecture[
                "generative_columns"
            ],
            "numerical_columns": architecture[
                "numerical_columns"
            ],
            "categorical_columns": architecture[
                "categorical_columns"
            ],
            "categorical_cardinalities": architecture[
                "categorical_cardinalities"
            ],
            "num_numerical": architecture[
                "num_numerical"
            ],
            "num_categorical": architecture[
                "num_categorical"
            ],
            "transformed_dim": architecture[
                "transformed_dim"
            ],
        }
        for dataset_name, architecture
        in SPPGAN_ARCHITECTURES.items()
    },

    "interfaces": {
        "loss": "SPPGANLossInterface",
        "statistical_guidance": "SPPGANStatisticalGuidance",
        "privacy": "SPPGANPrivacyInterface",
    },

    "downstream_notebooks": {
        "09": "SPP-GAN Statistical Guidance",
        "10": "SPP-GAN Differential Privacy",
        "11": "Privacy Accounting",
        "12": "SPP-GAN Training",
        "13": "Synthetic Data Generation",
        "14": "Statistical Fidelity Evaluation",
        "15": "ML Utility / TSTR Evaluation",
        "16": "Privacy & Disclosure Risk Evaluation",
        "17": "Ablation Study",
        "18": "Privacy–Fidelity–Utility Trade-off",
        "19": "Statistical Significance Testing",
        "20": "Final Comparative Analysis",
        "21": "Publication Tables & Figures",
        "22": "Reproducibility, Manifest & Final Audit",
    },

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

MODEL_CONFIG_PATH = (
    DIRS["config"]
    / "sppgan_model_configuration.json"
)

with open(
    MODEL_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        MODEL_CONFIGURATION,
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

print(f"✓ Model configuration saved:")
print(f"  {MODEL_CONFIG_PATH}")

23. SAVE MODEL CONFIGURATION
✓ Model configuration saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/config/sppgan_model_configuration.json


In [25]:
# ==================================================================================================
# 24. COMPLETION SUMMARY
# ==================================================================================================

print("=" * 100)
print("24. COMPLETION SUMMARY")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# Validation status
# -----------------------------------------------------------------------------------------------

shape_pass = (
    SHAPE_TEST_DF["shape_test"]
    .eq("PASS")
    .all()
)

forward_pass = (
    FORWARD_TEST_DF["forward_test"]
    .eq("PASS")
    .all()
)

backward_pass = (
    BACKWARD_TEST_DF["backward_test"]
    .eq("PASS")
    .all()
)

gradient_pass = (
    GRADIENT_VALIDATION_DF["gradient_test"]
    .eq("PASS")
    .all()
)

all_validation_pass = all([
    shape_pass,
    forward_pass,
    backward_pass,
    gradient_pass,
])

if not all_validation_pass:
    raise RuntimeError(
        "Notebook 08 validation failed."
    )

# -----------------------------------------------------------------------------------------------
# Completion manifest
# -----------------------------------------------------------------------------------------------

COMPLETION_MANIFEST = {
    "notebook": NOTEBOOK_ID,
    "name": NOTEBOOK_NAME,
    "framework": FRAMEWORK_NAME,

    "status": "PASS",

    "project_root": str(PROJECT_ROOT),

    "datasets_registered": len(
        SPPGAN_ARCHITECTURES
    ),

    "datasets": list(
        SPPGAN_ARCHITECTURES.keys()
    ),

    "architecture_validation": {
        "tensor_shape_tests": "PASS",
        "forward_pass_tests": "PASS",
        "backward_pass_tests": "PASS",
        "gradient_validation": "PASS",
    },

    "training_performed": False,
    "privacy_enabled": False,
    "synthetic_generation_performed": False,

    "artifacts": {
        "architecture_summary": str(
            summary_path
        ),
        "parameter_count": str(
            parameter_path
        ),
        "shape_tests": str(
            shape_path
        ),
        "forward_tests": str(
            forward_path
        ),
        "backward_tests": str(
            backward_path
        ),
        "gradient_validation": str(
            gradient_path
        ),
        "artifact_registry": str(
            artifact_path
        ),
        "model_configuration": str(
            MODEL_CONFIG_PATH
        ),
    },

    "next_notebook": {
        "number": "09",
        "name": "SPP-GAN Statistical Guidance",
    },

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

COMPLETION_PATH = (
    DIRS["validation"]
    / "sppgan_notebook_08_completion.json"
)

with open(
    COMPLETION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        COMPLETION_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

# -----------------------------------------------------------------------------------------------
# Final display
# -----------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 08 — FINAL STATUS")
print("=" * 100)

print(f"Framework                 : {FRAMEWORK_NAME}")
print(f"Datasets                  : {len(SPPGAN_ARCHITECTURES)}")
print(f"Tensor shape tests        : PASS")
print(f"Forward-pass tests        : PASS")
print(f"Backward-pass tests       : PASS")
print(f"Gradient validation       : PASS")
print(f"Training                  : NOT PERFORMED")
print(f"Privacy                   : NOT ENABLED")
print(f"Synthetic generation      : NOT PERFORMED")
print(f"Overall status            : PASS")

print("\nArtifacts:")
print(f"  Architecture : {DIRS['architecture']}")
print(f"  Configuration: {DIRS['config']}")
print(f"  Models       : {DIRS['models']}")
print(f"  Metadata     : {DIRS['metadata']}")
print(f"  Validation   : {DIRS['validation']}")

print("\nNext:")
print("  Notebook 09 — SPP-GAN Statistical Guidance")

print("=" * 100)

print("\n✓ NOTEBOOK 08 COMPLETED SUCCESSFULLY.")

24. COMPLETION SUMMARY

NOTEBOOK 08 — FINAL STATUS
Framework                 : SPP-GAN
Datasets                  : 3
Tensor shape tests        : PASS
Forward-pass tests        : PASS
Backward-pass tests       : PASS
Gradient validation       : PASS
Training                  : NOT PERFORMED
Privacy                   : NOT ENABLED
Synthetic generation      : NOT PERFORMED
Overall status            : PASS

Artifacts:
  Architecture : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/architecture
  Configuration: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/config
  Models       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/models
  Metadata     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/metadata
  Validation   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/validation

Next:
  Notebook 09 — SPP-GAN Statistical Guidance

✓ NOTEBOOK 08 COMPLETED SUCCESSFULLY.
